# Data preparation

In [1]:
import os
import ast
import pandas as pd
from rapidfuzz import fuzz
from torch_geometric.data import HeteroData
import torch

# ── Paths ────────────────────────────────────────────────────────────────────

BASE_DIR = os.path.abspath("..")
REAL_PATH   = os.path.join(BASE_DIR, "data", "raw", "EU_C12N15_data.csv")
SAMPLE_PATH = os.path.join(BASE_DIR, "data", "sample.csv")

for path in (REAL_PATH, SAMPLE_PATH):
    if os.path.exists(path):
        DATA_PATH = path
        break
else:
    raise FileNotFoundError(f"No dataset found.\nChecked:\n- {REAL_PATH}\n- {SAMPLE_PATH}")

raw_data = pd.read_csv(DATA_PATH)
print("Loaded:", DATA_PATH)
print("Shape:", raw_data.shape)

# ── Parsing ───────────────────────────────────────────────────────────────────

LIST_COLUMNS = ["Applicants", "Inventors", "CPC Classifications", "Priority Numbers"]

def parse_list_column(x):
    if pd.isna(x):
        return []
    if isinstance(x, list):
        return x
    try:
        return ast.literal_eval(x)
    except Exception:
        return [i.strip() for i in str(x).split(";")]

df = raw_data[raw_data["Publication Year"] < 2025].copy()
for col in LIST_COLUMNS:
    df[col] = df[col].apply(parse_list_column)

# ── Applicant name normalisation ──────────────────────────────────────────────

TOKEN_REPLACEMENTS = {
    "biotechnologies": "biotechnology",
    "technologies":    "technology",
    "therapeutics":    "therapeutic",
    "laboratories":    "laboratory",
    "systems":         "system",
    "sciences":        "science",
    "holdings":        "holding",
    "communications":  "communication",
    "solutions":       "solution",
}

def normalize_applicant_name(name):
    if pd.isna(name):
        return ""
    name = str(name).lower().strip()
    for char in ("&", ",", "."):
        name = name.replace(char, " " if char == "&" else " ")
    name = " ".join(name.split())  # collapse whitespace
    tokens = [TOKEN_REPLACEMENTS.get(tok, tok) for tok in name.split()]
    return " ".join(tokens)

def fuzzy_merge_applicants(applicant_names, threshold=93):
    """Map each name to a canonical form via fuzzy deduplication."""
    applicant_names = sorted(set(a for a in applicant_names if a))
    canonical_map, canonicals, merge_pairs = {}, [], []

    for name in applicant_names:
        for canon in canonicals:
            if (score := fuzz.ratio(name, canon)) >= threshold:
                canonical_map[name] = canon
                merge_pairs.append((name, canon, score))
                break
        else:
            canonical_map[name] = name
            canonicals.append(name)

    return canonical_map, merge_pairs

df["Applicants"] = df["Applicants"].apply(
    lambda lst: [normalize_applicant_name(a) for a in lst]
)

all_applicants_raw = sorted(set(a for row in df["Applicants"] for a in row if a))
applicant_name_map, merge_pairs = fuzzy_merge_applicants(all_applicants_raw, threshold=93)

df["Applicants"] = df["Applicants"].apply(
    lambda lst: [applicant_name_map.get(a, a) for a in lst]
)

print(f"Unique applicants before fuzzy merge: {len(all_applicants_raw)}")
print(f"Unique applicants after fuzzy merge:  {len(set(applicant_name_map.values()))}")
print(f"Number of merges made:                {len(merge_pairs)}")

merge_log_df = pd.DataFrame(merge_pairs, columns=["variant", "canonical", "score"])
display(merge_log_df.sort_values(["canonical", "score"], ascending=[True, False]).head(50))
merge_log_df.to_csv(os.path.join(BASE_DIR, "data", "processed", "applicant_name_merges.csv"), index=False)

# ── Global node registry ──────────────────────────────────────────────────────

def ensure_list(x):
    if isinstance(x, list): return x
    if pd.isna(x):          return []
    return [x]

for col in LIST_COLUMNS:
    df[col] = df[col].apply(ensure_list)

def make_mapping(items):
    return {v: i for i, v in enumerate(sorted(set(items)))}

applicant_to_id = make_mapping(a for row in df["Applicants"]         for a in row)
inventor_to_id  = make_mapping(i for row in df["Inventors"]          for i in row)
cpc_to_id       = make_mapping(c for row in df["CPC Classifications"] for c in row)
patent_to_id    = make_mapping(
    p for _, row in df.iterrows()
    for p in (row["Priority Numbers"] or [row["Lens ID"]])
)

nodes_df = pd.DataFrame({"node_id": applicant_to_id.values(), "name": applicant_to_id.keys()})
nodes_df.to_csv(os.path.join(BASE_DIR, "data", "processed", "metadata", "nodes_master.csv"), index=False)

# ── Graph construction ────────────────────────────────────────────────────────

def to_edge_index(edges):
    if not edges:
        return torch.empty((2, 0), dtype=torch.long)
    return torch.tensor(edges, dtype=torch.long).t().contiguous()

def add_edge_type(data, src_type, rel, dst_type, edges):
    ei = to_edge_index(edges)
    data[src_type, rel,           dst_type].edge_index = ei
    data[dst_type, f"rev_{rel}",  src_type].edge_index = to_edge_index([[j, i] for i, j in edges])

def build_pyg_hetero_graph(df, applicant_to_id, inventor_to_id, patent_to_id, cpc_to_id):
    data = HeteroData()
    app_pat_edges, inv_pat_edges, cpc_pat_edges = [], [], []

    for _, row in df.iterrows():
        patents = row["Priority Numbers"] or [row["Lens ID"]]
        for p in patents:
            if p not in patent_to_id:
                continue
            p_id = patent_to_id[p]
            for a in row["Applicants"]:
                if a in applicant_to_id:
                    app_pat_edges.append([applicant_to_id[a], p_id])
            for inv in row["Inventors"]:
                if inv in inventor_to_id:
                    inv_pat_edges.append([inventor_to_id[inv], p_id])
            for c in row["CPC Classifications"]:
                if c in cpc_to_id:
                    cpc_pat_edges.append([cpc_to_id[c], p_id])

    add_edge_type(data, "applicant", "files",      "patent", app_pat_edges)
    add_edge_type(data, "inventor",  "invented",   "patent", inv_pat_edges)
    add_edge_type(data, "cpc",       "in_patent",  "patent", cpc_pat_edges)

    data["applicant"].num_nodes = len(applicant_to_id)
    data["inventor"].num_nodes  = len(inventor_to_id)
    data["patent"].num_nodes    = len(patent_to_id)
    data["cpc"].num_nodes       = len(cpc_to_id)

    return data

# ── Build per-year graphs ─────────────────────────────────────────────────────

data_by_year = {
    year: build_pyg_hetero_graph(
        df[df["Publication Year"] == year],
        applicant_to_id, inventor_to_id, patent_to_id, cpc_to_id
    )
    for year in sorted(df["Publication Year"].dropna().unique())
}

data_by_year_cumulative = {
    year: build_pyg_hetero_graph(
        df[df["Publication Year"] <= year],   # <-- cumulative
        applicant_to_id, inventor_to_id, patent_to_id, cpc_to_id
    )
    for year in sorted(df["Publication Year"].dropna().unique())
}

for year, data in data_by_year.items():
    assert data["applicant"].num_nodes == len(applicant_to_id)
    assert data["patent"].num_nodes    == len(patent_to_id)

print("All yearly graphs share the same node space")

Loaded: /Users/carlos38/Desktop/Profesional/Master/EU_collabo_project/data/raw/EU_C12N15_data.csv
Shape: (3180, 14)
Unique applicants before fuzzy merge: 4516
Unique applicants after fuzzy merge:  4399
Number of merges made:                117


,variant,canonical,score
96,tron—translationale onkologie an der univ der ...,03 tron—translationale onkologie an der univ d...,96.511628
0,academisch ziekenhuis leiden a,academisch ziekenhuis leiden,96.551724
1,academisch ziekenhuis leiden h,academisch ziekenhuis leiden,96.551724
2,adiutide pharmaceuticals gmbh,adiu tide pharmaceuticals gmbh,98.305085
3,baeuerle patrick a,baeuerle patrick,94.117647
4,bahrami sherwin,bahrami shervin,93.333333
5,belgian volition srl,belgian volition sprl,97.560976
6,biocartis nv,biocartis n v,96.000000
7,brummelkamp thijn r,brummelkamp thijn,94.444444
8,bullerdiek jorn,bullerdiek joern,96.774194


All yearly graphs share the same node space


In [8]:
display(merge_log_df.sort_values(["score"], ascending=[True]).head(50))

,variant,canonical,score
82,schlingensiepen relmar,schlinensiepen reimar,93.023256
44,helmholtz zentrum muenchen deutsches forschung...,helmholtz zentrum m nchen deutsches forschungs...,93.167702
32,fundacio institut d investig biomedica de bell...,fundacio inst d investigacio biomedica de bell...,93.220339
4,bahrami sherwin,bahrami shervin,93.333333
76,polyplus transfection sa,polyplus transfection,93.333333
65,leturcq didier j,leturcq didier,93.333333
52,inserm institute national de la sante et de la...,inserm (institut naational de la santé et de l...,93.442623
85,st europeo di oncologia s r l,ieo st europeo di oncologia s r l,93.548387
73,ose immunotherapeutics sa,ose immunotherapeutics,93.617021
101,univ devry val dessone,univ d'evry val d'essonne,93.617021


# Feature Engineering

In [2]:
import torch
from torch_geometric.nn import MetaPath2Vec

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. MetaPaths (STRUCTURAL SIGNAL ONLY)
METAPATHS = {
    "APA": {
        "path": [
            ("applicant","files","patent"),
            ("patent","rev_files","applicant")
        ],
        "weight": 1.0
    },
    "API": {
        "path": [
            ("applicant","files","patent"),
            ("patent","rev_invented","inventor"),
            ("inventor","invented","patent"),
            ("patent","rev_files","applicant")
        ],
        "weight": 1.2
    },
    "APC": {
        "path": [
            ("applicant","files","patent"),
            ("patent","rev_in_patent","cpc"),
            ("cpc","in_patent","patent"),
            ("patent","rev_files","applicant")
        ],
        "weight": 1.2
    }
}


# 2. MetaPath2Vec training
def train_metapath_embedding(data, metapath, emb_dim=32, epochs=2):

    num_nodes = data["applicant"].num_nodes

    try:
        model = MetaPath2Vec(
            data.edge_index_dict,
            embedding_dim=emb_dim,
            metapath=metapath,
            walk_length=30,
            context_size=5,
            walks_per_node=3,
            num_negative_samples=5,
            sparse=True
        ).to(device)

        optimizer = torch.optim.SparseAdam(model.parameters(), lr=0.01)
        loader = model.loader(batch_size=128, shuffle=True)

        model.train()

        for _ in range(epochs):
            for pos_rw, neg_rw in loader:

                pos_rw = pos_rw.to(device)
                neg_rw = neg_rw.to(device)

                optimizer.zero_grad()
                loss = model.loss(pos_rw, neg_rw)

                # Safety against instability
                if torch.isnan(loss):
                    continue

                loss.backward()
                optimizer.step()

        model.eval()

        with torch.no_grad():
            emb = model("applicant").cpu()

        emb = torch.nan_to_num(emb, nan=0.0, posinf=5.0, neginf=-5.0)

    except Exception:
        # Fallback if metapath fails
        emb = torch.zeros(num_nodes, emb_dim)

    if emb.shape[0] != num_nodes:
        full_emb = torch.zeros(num_nodes, emb_dim)
        full_emb[:emb.shape[0]] = emb
        emb = full_emb

    return emb


# 3. Normalizing embeddings
def normalize_embeddings(z):
    return z / (z.norm(dim=1, keepdim=True) + 1e-8)


# 4. Generating embeddings per year
def generate_temporal_embeddings(data_by_year, metapaths=METAPATHS):

    emb_by_year = {}

    for year, data in data_by_year.items():

        emb_list = []

        for name, cfg in metapaths.items():

            emb = train_metapath_embedding(data, cfg["path"])

            # Apply importance weighting
            emb = emb * cfg["weight"]

            emb_list.append(emb)

        # Concatenate all metapath embeddings
        z = torch.cat(emb_list, dim=1)

        # Normalize for stability + similarity learning
        z = normalize_embeddings(z)

        emb_by_year[year] = z

    return emb_by_year


# 5. Build temporal tensor (T x N x D)
def build_temporal_tensor(emb_by_year, applicant_to_id):

    years_sorted = sorted(emb_by_year.keys())

    T = len(years_sorted)
    N = len(applicant_to_id)
    D = next(iter(emb_by_year.values())).shape[1]

    X = torch.zeros(T, N, D)

    for t, year in enumerate(years_sorted):
        X[t] = emb_by_year[year]

    return X, years_sorted


# 6. Debug graph structure
for year, data in data_by_year.items():
    print(f"{year}: {list(data.edge_index_dict.keys())}")


# 7. Run feature pipeline
emb_by_year = generate_temporal_embeddings(data_by_year)

X, years_sorted = build_temporal_tensor(emb_by_year, applicant_to_id)

print("✅ Feature tensor shape:", X.shape)  # (T, N, D)

1981: [('applicant', 'files', 'patent'), ('patent', 'rev_files', 'applicant'), ('inventor', 'invented', 'patent'), ('patent', 'rev_invented', 'inventor'), ('cpc', 'in_patent', 'patent'), ('patent', 'rev_in_patent', 'cpc')]
1983: [('applicant', 'files', 'patent'), ('patent', 'rev_files', 'applicant'), ('inventor', 'invented', 'patent'), ('patent', 'rev_invented', 'inventor'), ('cpc', 'in_patent', 'patent'), ('patent', 'rev_in_patent', 'cpc')]
1984: [('applicant', 'files', 'patent'), ('patent', 'rev_files', 'applicant'), ('inventor', 'invented', 'patent'), ('patent', 'rev_invented', 'inventor'), ('cpc', 'in_patent', 'patent'), ('patent', 'rev_in_patent', 'cpc')]
1985: [('applicant', 'files', 'patent'), ('patent', 'rev_files', 'applicant'), ('inventor', 'invented', 'patent'), ('patent', 'rev_invented', 'inventor'), ('cpc', 'in_patent', 'patent'), ('patent', 'rev_in_patent', 'cpc')]
1986: [('applicant', 'files', 'patent'), ('patent', 'rev_files', 'applicant'), ('inventor', 'invented', 'pat

# Model for visualization

In [4]:
# TEMPORAL GNN + SUPERVISED TEMPORAL LINK PREDICTION + EXPORT
# SNAPSHOT-BASED TEMPORAL GNN VERSION

import torch
import torch.nn as nn
import torch.nn.functional as F
import random
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score
from torch_geometric.utils import negative_sampling, add_self_loops
from torch_geometric.nn import GCNConv
import json
import os

# GLOBAL PATHS
BASE_DIR = os.path.abspath("..")
DATA_DIR = os.path.join(BASE_DIR, "data", "processed")

os.makedirs(os.path.join(DATA_DIR, "edges"), exist_ok=True)
os.makedirs(os.path.join(DATA_DIR, "metadata"), exist_ok=True)
os.makedirs(os.path.join(DATA_DIR, "embeddings"), exist_ok=True)
os.makedirs(os.path.join(DATA_DIR, "predictions"), exist_ok=True)

print("BASE_DIR:", BASE_DIR)
print("DATA_DIR:", DATA_DIR)

# SEED
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

# REQUIRED OBJECTS
required_vars = ["X", "data_by_year", "applicant_to_id"]
missing = [v for v in required_vars if v not in globals()]
if missing:
    raise ValueError(f"Missing required objects: {missing}")

global_apps = applicant_to_id
num_global_nodes = len(global_apps)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# HELPERS
def canonicalize_edge_index(edge_index: torch.Tensor) -> torch.Tensor:
    if edge_index is None or edge_index.numel() == 0:
        return torch.empty((2, 0), dtype=torch.long)

    arr = edge_index.t().cpu().numpy().astype(np.int64)
    arr = arr[arr[:, 0] != arr[:, 1]]
    if len(arr) == 0:
        return torch.empty((2, 0), dtype=torch.long)

    arr = np.sort(arr, axis=1)
    arr = np.unique(arr, axis=0)

    if len(arr) == 0:
        return torch.empty((2, 0), dtype=torch.long)

    return torch.tensor(arr, dtype=torch.long).t().contiguous()


def to_undirected_edge_index(edge_index: torch.Tensor) -> torch.Tensor:
    if edge_index is None or edge_index.numel() == 0:
        return torch.empty((2, 0), dtype=torch.long)

    rev = edge_index[[1, 0], :]
    out = torch.cat([edge_index, rev], dim=1)
    return canonicalize_edge_index(out)


def get_edge_index_from_store(data, edge_type):
    if edge_type not in data.edge_types:
        return None

    store = data[edge_type]

    if "edge_index" in store and store["edge_index"] is not None:
        return store["edge_index"].long()

    if "edge_label_index" in store and store["edge_label_index"] is not None:
        return store["edge_label_index"].long()

    if "adj_t" in store and store["adj_t"] is not None:
        row, col, _ = store["adj_t"].t().coo()
        return torch.stack([row, col], dim=0).long()

    return None


def find_applicant_patent_edge_types(data):
    files_type = None
    rev_files_type = None

    preferred_forward = []
    preferred_reverse = []
    fallback_forward = []
    fallback_reverse = []

    for et in data.edge_types:
        src, rel, dst = et
        rel_l = str(rel).lower()

        if src == "applicant" and dst == "patent":
            fallback_forward.append(et)
            if "file" in rel_l:
                preferred_forward.append(et)

        if src == "patent" and dst == "applicant":
            fallback_reverse.append(et)
            if "file" in rel_l:
                preferred_reverse.append(et)

    if preferred_forward:
        files_type = preferred_forward[0]
    elif fallback_forward:
        files_type = fallback_forward[0]

    if preferred_reverse:
        rev_files_type = preferred_reverse[0]
    elif fallback_reverse:
        rev_files_type = fallback_reverse[0]

    return files_type, rev_files_type


def infer_year_app_map(year, data, global_apps):
    if "app_maps" in globals() and year in app_maps:
        return app_maps[year]

    num_local = int(data["applicant"].num_nodes)

    candidate_keys = [
        "name", "names", "applicant_name", "applicant_names",
        "label", "labels", "id", "ids"
    ]

    for key in candidate_keys:
        if key in data["applicant"]:
            values = data["applicant"][key]

            if torch.is_tensor(values):
                values = values.cpu().tolist()
            else:
                values = list(values)

            inferred = {}
            ok = True

            for i, v in enumerate(values):
                if isinstance(v, bytes):
                    v = v.decode("utf-8", errors="ignore")
                v = str(v)

                if v not in global_apps:
                    ok = False
                    break

                inferred[v] = i

            if ok and len(inferred) > 0:
                print(f"Using inferred applicant mapping from '{key}' for year {year}")
                return inferred

    if num_local <= len(global_apps):
        print(f"Using identity applicant mapping fallback for year {year}")
        return {i: i for i in range(num_local)}

    raise ValueError(
        f"Could not infer applicant mapping for year {year}. "
        f"Local applicant nodes: {num_local}, global applicants: {len(global_apps)}"
    )


def map_local_applicant_id_to_global(local_id, app_map, global_map):
    if all(isinstance(k, (int, np.integer)) for k in app_map.keys()):
        return app_map.get(local_id, None)

    inv_app_map = {v: k for k, v in app_map.items()}
    applicant_key = inv_app_map.get(local_id, None)
    if applicant_key is None:
        return None
    return global_map.get(applicant_key, None)


def reconstruct_global_collab_edges_from_patents(year, data, app_map, global_map):
    files_type, rev_files_type = find_applicant_patent_edge_types(data)

    if files_type is None and rev_files_type is None:
        raise ValueError(
            f"Could not find applicant-patent relations for year {year}. "
            f"Available edge types: {data.edge_types}"
        )

    app_pat_parts = []

    if files_type is not None:
        ei = get_edge_index_from_store(data, files_type)
        if ei is not None and ei.numel() > 0:
            app_pat_parts.append(ei)

    if rev_files_type is not None:
        ei = get_edge_index_from_store(data, rev_files_type)
        if ei is not None and ei.numel() > 0:
            app_pat_parts.append(torch.stack([ei[1], ei[0]], dim=0))

    if len(app_pat_parts) == 0:
        return torch.empty((2, 0), dtype=torch.long)

    app_pat = torch.cat(app_pat_parts, dim=1)
    app_pat = torch.unique(app_pat.t(), dim=0).t().contiguous()

    patent_to_global_apps = {}

    for local_app_id, patent_id in app_pat.t().tolist():
        global_app_id = map_local_applicant_id_to_global(local_app_id, app_map, global_map)
        if global_app_id is None:
            continue
        if global_app_id < 0 or global_app_id >= len(global_map):
            continue

        patent_to_global_apps.setdefault(patent_id, set()).add(global_app_id)

    collab_edges = set()

    for _, app_set in patent_to_global_apps.items():
        apps = sorted(app_set)
        if len(apps) < 2:
            continue

        for i in range(len(apps)):
            for j in range(i + 1, len(apps)):
                a, b = apps[i], apps[j]
                if a != b:
                    x, y = (a, b) if a < b else (b, a)
                    collab_edges.add((x, y))

    if len(collab_edges) == 0:
        print(f"Year {year}: reconstructed collaboration edges are empty")
        return torch.empty((2, 0), dtype=torch.long)

    arr = np.array(list(collab_edges), dtype=np.int64)
    return torch.tensor(arr, dtype=torch.long).t().contiguous()


def build_yearly_global_graphs(data_by_year, years, global_apps):
    yearly_edges = {}
    for y in years:
        year_app_map = infer_year_app_map(y, data_by_year[y], global_apps)
        edge_index = reconstruct_global_collab_edges_from_patents(
            year=y,
            data=data_by_year[y],
            app_map=year_app_map,
            global_map=global_apps
        )
        edge_index = canonicalize_edge_index(edge_index)
        edge_index = to_undirected_edge_index(edge_index)
        edge_index, _ = add_self_loops(edge_index, num_nodes=len(global_apps))
        yearly_edges[y] = edge_index.long()
        print(f"Year {y}: graph edges = {tuple(yearly_edges[y].shape)}")
    return yearly_edges


# TEMPORAL GNN MODEL
class SnapshotTemporalGNN(nn.Module):
    def __init__(
        self,
        in_dim,
        gnn_hidden_dim=128,
        rnn_hidden_dim=128,
        gnn_layers=2,
        dropout=0.2,
        use_attention=True,
        predictor="mlp"
    ):
        super().__init__()

        self.use_attention = use_attention
        self.predictor = predictor
        self.dropout = dropout

        self.input_proj = nn.Linear(in_dim, gnn_hidden_dim)

        self.gnn_convs = nn.ModuleList()
        self.gnn_convs.append(GCNConv(gnn_hidden_dim, gnn_hidden_dim))
        for _ in range(gnn_layers - 1):
            self.gnn_convs.append(GCNConv(gnn_hidden_dim, gnn_hidden_dim))

        self.gru = nn.GRU(
            input_size=gnn_hidden_dim,
            hidden_size=rnn_hidden_dim,
            num_layers=1
        )

        self.att_proj = nn.Linear(rnn_hidden_dim, 1)

        self.mlp = nn.Sequential(
            nn.Linear(rnn_hidden_dim * 4, rnn_hidden_dim),
            nn.LeakyReLU(negative_slope=0.1),
            nn.Dropout(dropout),
            nn.Linear(rnn_hidden_dim, 1)
        )

    def encode_snapshot(self, x_t, edge_index_t):
        h = self.input_proj(x_t)

        for i, conv in enumerate(self.gnn_convs):
            h = conv(h, edge_index_t)
            if i < len(self.gnn_convs) - 1:
                h = F.relu(h)
                h = F.dropout(h, p=self.dropout, training=self.training)

        h = torch.nan_to_num(h, nan=0.0, posinf=5.0, neginf=-5.0)
        return h

    def encode(self, X_seq, edge_index_seq):
        snapshot_embs = []

        for t in range(X_seq.size(0)):
            h_t = self.encode_snapshot(X_seq[t], edge_index_seq[t])
            snapshot_embs.append(h_t)

        H = torch.stack(snapshot_embs, dim=0)   # [T, N, H_gnn]
        H_temporal, _ = self.gru(H)             # [T, N, H_rnn]

        if self.use_attention:
            att = torch.clamp(self.att_proj(H_temporal), -10, 10)
            scores = torch.softmax(att, dim=0)
            z = (H_temporal * scores).sum(dim=0)
        else:
            z = H_temporal.mean(dim=0)

        z = torch.nan_to_num(z, nan=0.0, posinf=5.0, neginf=-5.0)
        return H_temporal, z

    def decode_logits(self, z, edge_index):
        if edge_index.numel() == 0:
            return torch.empty(0, device=z.device)

        src, dst = edge_index
        s = z[src]
        d = z[dst]

        if self.predictor == "dot":
            return (s * d).sum(dim=1)

        feat = torch.cat([
            s,
            d,
            torch.abs(s - d),
            s * d
        ], dim=1)

        return self.mlp(feat).view(-1)

    def decode_proba(self, z, edge_index, clip=8.0):
        logits = self.decode_logits(z, edge_index)
        logits = torch.clamp(logits, -clip, clip)
        return torch.sigmoid(logits)


# TEMPORAL SPLIT
years = sorted(data_by_year.keys())
if len(years) < 2:
    raise ValueError("Need at least two years for temporal split.")

train_years = years[:-1]
test_year = years[-1]
train_T = len(train_years)

print("Train years:", train_years)
print("Test year:", test_year)

# NORMALIZE FEATURES
X = torch.nan_to_num(X, nan=0.0, posinf=1.0, neginf=-1.0).float()

X_train_window = X[:train_T]
mean = X_train_window.mean(dim=(0, 1), keepdim=True)
std = X_train_window.std(dim=(0, 1), keepdim=True)
std = torch.where(std < 1e-6, torch.ones_like(std), std)

X = (X - mean) / std
X = torch.nan_to_num(X, nan=0.0, posinf=5.0, neginf=-5.0)

# critical fix
X = X.detach().clone()
X_train = X[:train_T].detach().clone().to(device)
print("X_train shape:", tuple(X_train.shape))

# BUILD YEARLY APPLICANT GRAPHS
all_yearly_edges = build_yearly_global_graphs(data_by_year, years, global_apps)

train_edge_index_seq = [all_yearly_edges[y].to(device) for y in train_years]

train_edge_list = []
for y in train_years:
    e = canonicalize_edge_index(all_yearly_edges[y].cpu())
    if e.numel() > 0:
        train_edge_list.append(e)

if len(train_edge_list) == 0:
    raise ValueError("❌ No training collaboration edges were reconstructed from train years.")

train_edge_index = canonicalize_edge_index(torch.cat(train_edge_list, dim=1)).to(device)
if train_edge_index.numel() == 0:
    raise ValueError("❌ Training edge index is empty after cleaning.")

np.save(
    os.path.join(DATA_DIR, "edges", "train_edges.npy"),
    train_edge_index.cpu().numpy()
)
print("✅ Saved train edges:", tuple(train_edge_index.shape))

test_edge_index = canonicalize_edge_index(all_yearly_edges[test_year].cpu()).to(device)
if test_edge_index.numel() == 0:
    raise ValueError("❌ No test collaboration edges were reconstructed for the final year.")

np.save(
    os.path.join(DATA_DIR, "edges", "test_edges.npy"),
    test_edge_index.cpu().numpy()
)
print("✅ Saved test edges:", tuple(test_edge_index.shape))

# MODEL
model = SnapshotTemporalGNN(
    in_dim=X_train.shape[-1],
    gnn_hidden_dim=128,
    rnn_hidden_dim=128,
    gnn_layers=2,
    dropout=0.2,
    use_attention=True,
    predictor="mlp"
).to(device)

opt = torch.optim.Adam(model.parameters(), lr=5e-4, weight_decay=1e-4)
criterion = nn.BCEWithLogitsLoss()

losses = []
auc_history = []
ap_history = []

# TRAINING
epochs = 300

for epoch in range(epochs):
    model.train()
    opt.zero_grad(set_to_none=True)

    _, z_train = model.encode(X_train.detach(), train_edge_index_seq)

    pos_logits = model.decode_logits(z_train, train_edge_index)

    neg_edge_index = negative_sampling(
        edge_index=train_edge_index,
        num_nodes=z_train.size(0),
        num_neg_samples=train_edge_index.size(1),
        method="sparse"
    ).to(device)

    neg_logits = model.decode_logits(z_train, neg_edge_index)

    logits = torch.cat([pos_logits, neg_logits], dim=0)
    labels = torch.cat([
        torch.ones_like(pos_logits),
        torch.zeros_like(neg_logits)
    ], dim=0)

    emb_penalty = 0.001 * z_train.pow(2).mean()
    loss = criterion(logits, labels) + emb_penalty

    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step()

    losses.append(float(loss.item()))

    del z_train, pos_logits, neg_logits, logits, labels, neg_edge_index, emb_penalty, loss

    if epoch % 5 == 0 or epoch == epochs - 1:
        model.eval()
        with torch.no_grad():
            _, z_eval = model.encode(X_train.detach(), train_edge_index_seq)

            pos_pred = model.decode_proba(z_eval, test_edge_index)

            neg_test_edge_index = negative_sampling(
                edge_index=test_edge_index,
                num_nodes=z_eval.size(0),
                num_neg_samples=test_edge_index.size(1),
                method="sparse"
            ).to(device)

            neg_pred = model.decode_proba(z_eval, neg_test_edge_index)

            preds = torch.cat([pos_pred, neg_pred]).cpu().numpy()
            labels_eval = torch.cat([
                torch.ones(pos_pred.size(0), device=device),
                torch.zeros(neg_pred.size(0), device=device)
            ]).cpu().numpy()

            auc = roc_auc_score(labels_eval, preds)
            ap = average_precision_score(labels_eval, preds)

            auc_history.append((epoch, float(auc)))
            ap_history.append((epoch, float(ap)))

        print(
            f"Epoch {epoch:03d} | "
            f"Loss: {losses[-1]:.4f} | "
            f"Test AUC: {auc:.4f} | "
            f"Test AP: {ap:.4f}"
        )

# FINAL EVALUATION
model.eval()
with torch.no_grad():
    h_train, z_train = model.encode(X_train.detach(), train_edge_index_seq)

    pos_pred = model.decode_proba(z_train, test_edge_index)

    neg_test_edge_index = negative_sampling(
        edge_index=test_edge_index,
        num_nodes=z_train.size(0),
        num_neg_samples=test_edge_index.size(1),
        method="sparse"
    ).to(device)

    neg_pred = model.decode_proba(z_train, neg_test_edge_index)

    preds = torch.cat([pos_pred, neg_pred]).cpu().numpy()
    labels_eval = torch.cat([
        torch.ones(pos_pred.size(0), device=device),
        torch.zeros(neg_pred.size(0), device=device)
    ]).cpu().numpy()

    auc = roc_auc_score(labels_eval, preds)
    ap = average_precision_score(labels_eval, preds)

z = z_train.detach().cpu()
z_norm = z / (z.norm(dim=1, keepdim=True) + 1e-8)

np.save(os.path.join(DATA_DIR, "embeddings", "z_2024.npy"), z.numpy())

print("\nFINAL MODEL PERFORMANCE:")
print(f"AUC: {auc:.4f}, AP: {ap:.4f}")
print(f"Positive score max: {float(pos_pred.max()):.4f}")
print(f"Negative score max: {float(neg_pred.max()):.4f}")
print(f"Positive score mean: {float(pos_pred.mean()):.4f}")
print(f"Negative score mean: {float(neg_pred.mean()):.4f}")

BASE_DIR: /Users/carlos38/Desktop/Profesional/Master/EU_collabo_project
DATA_DIR: /Users/carlos38/Desktop/Profesional/Master/EU_collabo_project/data/processed
Using device: cpu
Train years: [1981, 1983, 1984, 1985, 1986, 1987, 1988, 1990, 1991, 1992, 1993, 1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023]
Test year: 2024
X_train shape: (41, 4400, 96)
Using identity applicant mapping fallback for year 1981
Year 1981: reconstructed collaboration edges are empty
Year 1981: graph edges = (2, 4400)
Using identity applicant mapping fallback for year 1983
Year 1983: reconstructed collaboration edges are empty
Year 1983: graph edges = (2, 4400)
Using identity applicant mapping fallback for year 1984
Year 1984: graph edges = (2, 4403)
Using identity applicant mapping fallback for year 1985
Year 1985: graph edges = (2, 4403)
Using identity applicant mapping fallback f

In [5]:
print("\nFINAL MODEL PERFORMANCE:")
print(f"AUC: {auc:.4f}, AP: {ap:.4f}")


FINAL MODEL PERFORMANCE:
AUC: 0.8555, AP: 0.8443


In [6]:
# SAVE METRICS
from sklearn.cluster import KMeans
def to_py(x):
    if isinstance(x, (np.integer,)):
        return int(x)
    if isinstance(x, (np.floating,)):
        return float(x)
    if isinstance(x, torch.Tensor):
        if x.numel() == 1:
            return x.item()
        return x.detach().cpu().tolist()
    if isinstance(x, list):
        return [to_py(v) for v in x]
    if isinstance(x, tuple):
        return [to_py(v) for v in x]
    if isinstance(x, dict):
        return {str(k): to_py(v) for k, v in x.items()}
    return x

metrics = {
    "losses": [float(v) for v in losses],
    "auc_history": [{"epoch": int(e), "auc": float(v)} for e, v in auc_history],
    "ap_history": [{"epoch": int(e), "ap": float(v)} for e, v in ap_history],
    "final_auc": float(auc),
    "final_ap": float(ap),
    "train_years": [int(y) for y in train_years],
    "test_year": int(test_year),
    "num_train_edges": int(train_edge_index.size(1)),
    "num_test_edges": int(test_edge_index.size(1)),
    "score_diagnostics": {
        "positive_max": float(pos_pred.max().item()),
        "positive_mean": float(pos_pred.mean().item()),
        "negative_max": float(neg_pred.max().item()),
        "negative_mean": float(neg_pred.mean().item())
    },
    "model_config": {
        "predictor": "mlp",
        "temporal": "attention",
        "dropout": 0.3,
        "activation": "LeakyReLU",
        "hidden_dim": 64
    }
}

metrics = to_py(metrics)

with open(os.path.join(DATA_DIR, "predictions", "training_metrics.json"), "w") as f:
    json.dump(metrics, f, indent=2)

# NODE METADATA
id_to_app = {i: app for app, i in global_apps.items()}

df_nodes = pd.DataFrame({
    "node_id": list(id_to_app.keys()),
    "applicant_name": list(id_to_app.values())
}).sort_values("node_id")

# EMBEDDINGS / CLUSTERS
k = 10
clusters = KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(z.numpy())
df_nodes["cluster"] = clusters

df_nodes.to_csv(
    os.path.join(DATA_DIR, "metadata", "nodes_with_clusters.csv"),
    index=False
)

np.save(os.path.join(DATA_DIR, "embeddings", "z_2024.npy"), z.numpy())
np.save(os.path.join(DATA_DIR, "embeddings", "z_norm.npy"), z_norm.numpy())

print("Saved nodes metadata and embeddings")

temporal_clusters = [
    KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(h_train[t].cpu().numpy())
    for t in range(h_train.shape[0])
]
np.save(
    os.path.join(DATA_DIR, "embeddings", "temporal_clusters.npy"),
    np.array(temporal_clusters)
)

cluster_matrix = np.zeros((k, k), dtype=np.float32)
for i in range(k):
    for j in range(k):
        mask_i = (clusters == i)
        mask_j = (clusters == j)
        if mask_i.sum() > 0 and mask_j.sum() > 0:
            sim_block = (z_norm[mask_i] @ z_norm[mask_j].T).numpy()
            cluster_matrix[i, j] = sim_block.mean()

np.save(
    os.path.join(DATA_DIR, "predictions", "cluster_matrix.npy"),
    cluster_matrix
)

# KNOWN EDGES
known_edges = set()

for src, dst in train_edge_index.t().tolist():
    a, b = (src, dst) if src < dst else (dst, src)
    known_edges.add((a, b))

for src, dst in test_edge_index.t().tolist():
    a, b = (src, dst) if src < dst else (dst, src)
    known_edges.add((a, b))

# TOP-K PREDICTED LINKS
top_links = {}
num_nodes = z.size(0)
TOP_K = 50

with torch.no_grad():
    for i in range(num_nodes):
        all_targets = torch.arange(num_nodes, dtype=torch.long)
        mask = all_targets != i
        targets = all_targets[mask]

        pair_index = torch.stack([
            torch.full_like(targets, i),
            targets
        ], dim=0)

        scores = model.decode_proba(z, pair_index).view(-1)

        scored = []
        for tgt, score in zip(targets.tolist(), scores.tolist()):
            a, b = (i, tgt) if i < tgt else (tgt, i)
            if (a, b) in known_edges:
                continue

            scored.append({
                "target": int(tgt),
                "score": float(score)
            })

        scored = sorted(scored, key=lambda x: x["score"], reverse=True)[:TOP_K]
        top_links[str(i)] = scored

with open(os.path.join(DATA_DIR, "predictions", "top_links.json"), "w") as f:
    json.dump(top_links, f)

print("Saved top predicted links")

# FULL PROBABILITY MATRIX
prob_matrix = torch.zeros((num_nodes, num_nodes))

with torch.no_grad():
    for i in range(num_nodes):
        all_targets = torch.arange(num_nodes, dtype=torch.long)
        pair_index = torch.stack([
            torch.full_like(all_targets, i),
            all_targets
        ], dim=0)

        scores = model.decode_proba(z, pair_index).view(-1)
        prob_matrix[i] = scores.cpu()

np.save(
    os.path.join(DATA_DIR, "predictions", "prob_matrix.npy"),
    prob_matrix.numpy()
)

print("Saved full probability matrix")
print("\nPIPELINE COMPLETE (METAPATH EMBEDDINGS + GRU TEMPORALITY + CO-FILING LABELS)")

Saved nodes metadata and embeddings
Saved top predicted links
Saved full probability matrix

PIPELINE COMPLETE (METAPATH EMBEDDINGS + GRU TEMPORALITY + CO-FILING LABELS)


# Ablation study

In [ ]:
# ABLATION STUDY:
# predictor = dot vs mlp
# temporal aggregation = attention vs mean
# metapath attention = True vs False

import torch
import torch.nn as nn
import torch.nn.functional as F
import random
import numpy as np
import pandas as pd
import json
import os
import itertools

from sklearn.metrics import roc_auc_score, average_precision_score
from torch_geometric.utils import negative_sampling

# PATHS
BASE_DIR = os.path.abspath("..")
DATA_DIR = os.path.join(BASE_DIR, "data", "processed")
os.makedirs(os.path.join(DATA_DIR, "predictions"), exist_ok=True)

# SEED
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# CHECK REQUIRED OBJECTS
required_vars = ["X", "data_by_year", "applicant_to_id"]
missing = [v for v in required_vars if v not in globals()]
if missing:
    raise ValueError(f"Missing required objects: {missing}")

global_apps = applicant_to_id

# METAPATH SLICES
# X was built as concatenation of:
# APA (32) + API (32) + APC (32) = 96 dims
META_FEAT_SLICES = {
    "APA": slice(0, 32),
    "API": slice(32, 64),
    "APC": slice(64, 96),
}

print("META_FEAT_SLICES:", META_FEAT_SLICES)

# MODEL
class TemporalLinkPredictor(nn.Module):
    def __init__(
        self,
        in_dim,
        hidden_dim=64,
        use_attention=True,
        use_metapath_attention=False,
        predictor="dot",
        dropout=0.3,
        meta_feat_slices=None
    ):
        super().__init__()

        self.use_attention = use_attention
        self.use_metapath_attention = use_metapath_attention
        self.predictor = predictor
        self.dropout = dropout
        self.meta_feat_slices = meta_feat_slices

        if self.use_metapath_attention:
            if meta_feat_slices is None or len(meta_feat_slices) < 2:
                raise ValueError("Metapath attention requires valid META_FEAT_SLICES.")

            self.meta_names = list(meta_feat_slices.keys())

            self.meta_proj = nn.ModuleDict({
                name: nn.Linear(
                    meta_feat_slices[name].stop - meta_feat_slices[name].start,
                    hidden_dim
                )
                for name in self.meta_names
            })

            self.meta_att_proj = nn.Linear(hidden_dim, 1)
            gru_input_dim = hidden_dim

        else:
            self.input_proj = nn.Linear(in_dim, hidden_dim)
            gru_input_dim = hidden_dim

        self.gru = nn.GRU(
            input_size=gru_input_dim,
            hidden_size=hidden_dim,
            num_layers=1
        )

        if self.use_attention:
            self.att_proj = nn.Linear(hidden_dim, 1)

        if self.predictor == "mlp":
            self.mlp = nn.Sequential(
                nn.Linear(hidden_dim * 4, hidden_dim),
                nn.LeakyReLU(negative_slope=0.1),
                nn.Dropout(dropout),
                nn.Linear(hidden_dim, 1)
            )

    def apply_metapath_attention(self, x_t):
        meta_embs = []
        meta_scores = []

        for name in self.meta_names:
            sl = self.meta_feat_slices[name]
            x_meta = x_t[:, sl]                       # [N, 32]
            h_meta = torch.tanh(self.meta_proj[name](x_meta))  # [N, H]
            s_meta = self.meta_att_proj(h_meta)      # [N, 1]

            meta_embs.append(h_meta)
            meta_scores.append(s_meta)

        H_meta = torch.stack(meta_embs, dim=1)       # [N, M, H]
        S_meta = torch.stack(meta_scores, dim=1)     # [N, M, 1]
        A_meta = torch.softmax(S_meta, dim=1)        # [N, M, 1]

        h = (H_meta * A_meta).sum(dim=1)             # [N, H]
        return h

    def encode(self, X):
        # X: [T, N, F]
        snapshot_list = []

        for t in range(X.size(0)):
            x_t = X[t]

            if self.use_metapath_attention:
                h_t = self.apply_metapath_attention(x_t)
            else:
                h_t = self.input_proj(x_t)

            h_t = torch.nan_to_num(h_t, nan=0.0, posinf=5.0, neginf=-5.0)
            snapshot_list.append(h_t)

        H_in = torch.stack(snapshot_list, dim=0)     # [T, N, H]
        h, _ = self.gru(H_in)                        # [T, N, H]

        if self.use_attention:
            att = torch.clamp(self.att_proj(h), -10, 10)
            scores = torch.softmax(att, dim=0)
            z = (h * scores).sum(dim=0)
        else:
            z = h.mean(dim=0)

        z = torch.nan_to_num(z, nan=0.0, posinf=5.0, neginf=-5.0)
        return h, z

    def decode_logits(self, z, edge_index):
        if edge_index.numel() == 0:
            return torch.empty(0, device=z.device)

        src, dst = edge_index
        s = z[src]
        d = z[dst]

        if self.predictor == "dot":
            return (s * d).sum(dim=1)

        feat = torch.cat([
            s,
            d,
            torch.abs(s - d),
            s * d
        ], dim=1)

        return self.mlp(feat).view(-1)

    def decode_proba(self, z, edge_index, clip=8.0):
        logits = self.decode_logits(z, edge_index)
        logits = torch.clamp(logits, -clip, clip)
        return torch.sigmoid(logits)

# TEMPORAL SPLIT
years = sorted(data_by_year.keys())
if len(years) < 2:
    raise ValueError("Need at least two years for temporal split.")

train_years = years[:-1]
test_year = years[-1]
train_T = len(train_years)

print("Train years:", train_years)
print("Test year:", test_year)

# NORMALIZATION
X = torch.nan_to_num(X, nan=0.0, posinf=1.0, neginf=-1.0).float()

X_train_window = X[:train_T]
mean = X_train_window.mean(dim=(0, 1), keepdim=True)
std = X_train_window.std(dim=(0, 1), keepdim=True)
std = torch.where(std < 1e-6, torch.ones_like(std), std)

X = (X - mean) / std
X = torch.nan_to_num(X, nan=0.0, posinf=5.0, neginf=-5.0)

# important: remove any old autograd history
X = X.detach().clone()
X_train = X[:train_T].detach().clone().to(device)

print("X_train shape:", tuple(X_train.shape))

# BUILD LABEL EDGES
train_edge_list = []

for y in train_years:
    year_app_map = infer_year_app_map(y, data_by_year[y], global_apps)
    global_edge = reconstruct_global_collab_edges_from_patents(
        year=y,
        data=data_by_year[y],
        app_map=year_app_map,
        global_map=global_apps
    )

    if global_edge.numel() > 0:
        train_edge_list.append(global_edge)

if len(train_edge_list) == 0:
    raise ValueError("No training collaboration edges were reconstructed from train years.")

train_edge_index = canonicalize_edge_index(torch.cat(train_edge_list, dim=1)).to(device)
if train_edge_index.numel() == 0:
    raise ValueError("Training edge index is empty after cleaning.")

test_app_map = infer_year_app_map(test_year, data_by_year[test_year], global_apps)
test_edge_index = reconstruct_global_collab_edges_from_patents(
    year=test_year,
    data=data_by_year[test_year],
    app_map=test_app_map,
    global_map=global_apps
)
test_edge_index = canonicalize_edge_index(test_edge_index).to(device)

if test_edge_index.numel() == 0:
    raise ValueError("No test collaboration edges were reconstructed for the final year.")

print("Train edges:", tuple(train_edge_index.shape))
print("Test edges:", tuple(test_edge_index.shape))

# TRAIN/EVAL FUNCTION
def run_experiment(
    X_train,
    train_edge_index,
    test_edge_index,
    predictor="dot",
    use_attention=True,
    use_metapath_attention=False,
    hidden_dim=64,
    dropout=0.3,
    epochs=100,
    lr=5e-4,
    weight_decay=1e-4,
    seed=42
):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    model = TemporalLinkPredictor(
        in_dim=X_train.shape[-1],
        hidden_dim=hidden_dim,
        use_attention=use_attention,
        use_metapath_attention=use_metapath_attention,
        predictor=predictor,
        dropout=dropout,
        meta_feat_slices=META_FEAT_SLICES
    ).to(device)

    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.BCEWithLogitsLoss()

    best_auc = -1.0
    best_ap = -1.0
    best_epoch = -1

    for epoch in range(epochs):
        model.train()
        opt.zero_grad(set_to_none=True)

        _, z_train = model.encode(X_train)

        pos_logits = model.decode_logits(z_train, train_edge_index)

        neg_edge_index = negative_sampling(
            edge_index=train_edge_index,
            num_nodes=z_train.size(0),
            num_neg_samples=train_edge_index.size(1),
            method="sparse"
        ).to(device)

        neg_logits = model.decode_logits(z_train, neg_edge_index)

        logits = torch.cat([pos_logits, neg_logits], dim=0)
        labels = torch.cat([
            torch.ones_like(pos_logits),
            torch.zeros_like(neg_logits)
        ], dim=0)

        emb_penalty = 0.001 * z_train.pow(2).mean()
        loss = criterion(logits, labels) + emb_penalty

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()

        model.eval()
        with torch.no_grad():
            _, z_eval = model.encode(X_train)

            pos_pred = model.decode_proba(z_eval, test_edge_index)

            neg_test_edge_index = negative_sampling(
                edge_index=test_edge_index,
                num_nodes=z_eval.size(0),
                num_neg_samples=test_edge_index.size(1),
                method="sparse"
            ).to(device)

            neg_pred = model.decode_proba(z_eval, neg_test_edge_index)

            preds = torch.cat([pos_pred, neg_pred]).cpu().numpy()
            labels_eval = torch.cat([
                torch.ones(pos_pred.size(0), device=device),
                torch.zeros(neg_pred.size(0), device=device)
            ]).cpu().numpy()

            auc = roc_auc_score(labels_eval, preds)
            ap = average_precision_score(labels_eval, preds)

            if auc > best_auc:
                best_auc = float(auc)
                best_ap = float(ap)
                best_epoch = epoch + 1

        del z_train, pos_logits, neg_edge_index, neg_logits, logits, labels, emb_penalty, loss

    # final evaluation
    model.eval()
    with torch.no_grad():
        _, z_final = model.encode(X_train)

        pos_pred = model.decode_proba(z_final, test_edge_index)

        neg_test_edge_index = negative_sampling(
            edge_index=test_edge_index,
            num_nodes=z_final.size(0),
            num_neg_samples=test_edge_index.size(1),
            method="sparse"
        ).to(device)

        neg_pred = model.decode_proba(z_final, neg_test_edge_index)

        preds = torch.cat([pos_pred, neg_pred]).cpu().numpy()
        labels_eval = torch.cat([
            torch.ones(pos_pred.size(0), device=device),
            torch.zeros(neg_pred.size(0), device=device)
        ]).cpu().numpy()

        final_auc = roc_auc_score(labels_eval, preds)
        final_ap = average_precision_score(labels_eval, preds)

    return {
        "metapath_attention": use_metapath_attention,
        "dynamic_attention": use_attention,
        "predictor": predictor,
        "hidden_dim": hidden_dim,
        "dropout": dropout,
        "epochs": epochs,
        "best_auc": float(best_auc),
        "best_ap": float(best_ap),
        "best_epoch": int(best_epoch),
        "final_auc": float(final_auc),
        "final_ap": float(final_ap),
    }

# ABLATION GRID
configs = list(itertools.product(
    [False, True],   # metapath attention
    [False, True],   # dynamic attention
    ["dot", "mlp"]   # predictor
))

results = []

print("\n=== ABLATION STUDY ===")
for use_meta, use_dyn, pred in configs:
    print(
        f"Running metapath_attention={use_meta} | "
        f"dynamic_attention={use_dyn} | "
        f"predictor={pred}"
    )

    out = run_experiment(
        X_train=X_train,
        train_edge_index=train_edge_index,
        test_edge_index=test_edge_index,
        predictor=pred,
        use_attention=use_dyn,
        use_metapath_attention=use_meta,
        hidden_dim=64,
        dropout=0.3,
        epochs=100,
        lr=5e-4,
        weight_decay=1e-4,
        seed=42
    )

    results.append(out)

    print(
        f" -> final AUC: {out['final_auc']:.4f} | "
        f"final AP: {out['final_ap']:.4f} | "
        f"best AUC: {out['best_auc']:.4f} @ epoch {out['best_epoch']}"
    )

# RESULTS TABLE
results_df = pd.DataFrame(results).sort_values(
    ["final_auc", "final_ap"],
    ascending=False
).reset_index(drop=True)

print("\n=== ABLATION RESULTS ===")
print(results_df)

results_df.to_csv(
    os.path.join(DATA_DIR, "predictions", "ablation_metapath_dynamic_predictor.csv"),
    index=False
)

with open(os.path.join(DATA_DIR, "predictions", "ablation_metapath_dynamic_predictor.json"), "w") as f:
    json.dump(results, f, indent=2)

print("\nSaved ablation results.")

Using device: cpu
META_FEAT_SLICES: {'APA': slice(0, 32, None), 'API': slice(32, 64, None), 'APC': slice(64, 96, None)}
Train years: [1981, 1983, 1984, 1985, 1986, 1987, 1988, 1990, 1991, 1992, 1993, 1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023]
Test year: 2024
X_train shape: (41, 4339, 96)
Using identity applicant mapping fallback for year 1981
Year 1981: reconstructed collaboration edges are empty
Using identity applicant mapping fallback for year 1983
Year 1983: reconstructed collaboration edges are empty
Using identity applicant mapping fallback for year 1984
Using identity applicant mapping fallback for year 1985
Using identity applicant mapping fallback for year 1986
Using identity applicant mapping fallback for year 1987
Using identity applicant mapping fallback for year 1988
Year 1988: reconstructed collaboration edges are empty
Using identity ap

In [38]:
results_df

,metapath_attention,dynamic_attention,predictor,hidden_dim,dropout,epochs,best_auc,best_ap,best_epoch,final_auc,final_ap
0,False,True,mlp,64,0.3,100,0.775896,0.741547,96,0.769635,0.737679
1,False,False,mlp,64,0.3,100,0.766663,0.732571,99,0.758717,0.725488
2,True,False,mlp,64,0.3,100,0.743187,0.713792,100,0.737190,0.708912
3,True,True,mlp,64,0.3,100,0.740091,0.713079,100,0.734864,0.707074
4,True,False,dot,64,0.3,100,0.564817,0.557820,100,0.561028,0.556766
5,True,True,dot,64,0.3,100,0.562622,0.554926,100,0.558828,0.553859
6,False,False,dot,64,0.3,100,0.562010,0.561735,55,0.550883,0.548908
7,False,True,dot,64,0.3,100,0.562438,0.563879,55,0.549249,0.551884


# Explanatory model

In [7]:
# ============================================================
# CELL A — WALK-LEVEL EXPLAINER SETUP
# Add after your current notebook cells
# ============================================================

import math
import torch
import pandas as pd
import torch.nn.functional as F
from collections import defaultdict
from torch_geometric.nn import MetaPath2Vec

# Reverse maps from your existing notebook objects:
id_to_applicant = {v: k for k, v in applicant_to_id.items()}
id_to_inventor  = {v: k for k, v in inventor_to_id.items()}
id_to_patent    = {v: k for k, v in patent_to_id.items()}
id_to_cpc       = {v: k for k, v in cpc_to_id.items()}

NODE_NAME_LOOKUP = {
    "applicant": id_to_applicant,
    "inventor": id_to_inventor,
    "patent": id_to_patent,
    "cpc": id_to_cpc,
}

EXPLAIN_CONTEXT_SIZE = 5          # must match your current MetaPath2Vec setup
EXPLAIN_WALK_LENGTH  = 30         # must match your current MetaPath2Vec setup
EXPLAIN_WALKS_PER_NODE = 3        # must match your current MetaPath2Vec setup
EXPLAIN_NEG_SAMPLES = 5           # must match your current MetaPath2Vec setup
EXPLAIN_EMB_DIM = 32              # matches your per-metapath embedding size
EXPLAIN_EPOCHS = 2                # matches your current notebook

# You can change this later:
EXPLAIN_YEAR = test_year if "test_year" in globals() else years_sorted[-1]

def safe_lookup(node_type, local_id):
    lookup = NODE_NAME_LOOKUP.get(node_type, {})
    return lookup.get(local_id, f"{node_type}_{local_id}")

def normalize_applicant_query(name):
    # reuses your notebook normalization if available
    try:
        norm = normalize_applicant_name(name)
        norm = applicant_name_map.get(norm, norm) if "applicant_name_map" in globals() else norm
        return norm
    except Exception:
        return name

def resolve_applicant_id(name_or_id):
    if isinstance(name_or_id, int):
        return name_or_id
    q = normalize_applicant_query(name_or_id)
    if q in applicant_to_id:
        return applicant_to_id[q]
    raise KeyError(f"Applicant not found: {name_or_id}")

def decode_global_token(token, mp2v_model):
    token = int(token)

    if token == int(mp2v_model.dummy_idx):
        return {
            "token": token,
            "node_type": "dummy",
            "local_id": None,
            "name": "<dummy>"
        }

    for node_type in mp2v_model.start.keys():
        start = int(mp2v_model.start[node_type])
        end   = int(mp2v_model.end[node_type])
        if start <= token < end:
            local_id = token - start
            return {
                "token": token,
                "node_type": node_type,
                "local_id": local_id,
                "name": safe_lookup(node_type, local_id)
            }

    return {
        "token": token,
        "node_type": "unknown",
        "local_id": None,
        "name": f"<unknown:{token}>"
    }

def decode_rw_row(row, mp2v_model):
    return [decode_global_token(tok, mp2v_model) for tok in row.tolist()]

def format_decoded_rw(decoded_row):
    parts = []
    for item in decoded_row:
        if item["node_type"] == "dummy":
            parts.append("<dummy>")
        else:
            parts.append(f'{item["node_type"]}:{item["name"]}')
    return " -> ".join(parts)

def build_type_embedding_cache(mp2v_model):
    cache = {}
    mp2v_model.eval()
    with torch.no_grad():
        for node_type in mp2v_model.start.keys():
            cache[node_type] = mp2v_model(node_type).detach().cpu()
    return cache

def walk_window_embedding(decoded_row, type_emb_cache):
    vecs = []
    for item in decoded_row:
        nt = item["node_type"]
        lid = item["local_id"]
        if nt in type_emb_cache and lid is not None:
            if 0 <= lid < type_emb_cache[nt].size(0):
                vecs.append(type_emb_cache[nt][lid])

    if not vecs:
        return None

    return torch.stack(vecs, dim=0).mean(dim=0)

def cosine_safe(a, b):
    a = a.view(1, -1)
    b = b.view(1, -1)
    return float(F.cosine_similarity(a, b).item())

def edge_index_to_set(edge_index):
    if edge_index is None or edge_index.numel() == 0:
        return set()
    out = set()
    for u, v in edge_index.t().cpu().tolist():
        if u == v:
            continue
        a, b = (u, v) if u < v else (v, u)
        out.add((a, b))
    return out

TRAIN_EDGE_SET = edge_index_to_set(train_edge_index.cpu()) if "train_edge_index" in globals() else set()
TEST_EDGE_SET  = edge_index_to_set(test_edge_index.cpu())  if "test_edge_index" in globals() else set()

print("Explainer setup ready.")
print("Default explain year:", EXPLAIN_YEAR)

Explainer setup ready.
Default explain year: 2024


In [8]:
# ============================================================
# CELL B — TRAIN AN EXPLAINABLE CLONE OF METAPATH2VEC
# This does not modify your original pipeline
# ============================================================

def train_metapath_embedding_with_walk_windows(
    data,
    metapath,
    emb_dim=EXPLAIN_EMB_DIM,
    epochs=EXPLAIN_EPOCHS,
    walk_length=EXPLAIN_WALK_LENGTH,
    context_size=EXPLAIN_CONTEXT_SIZE,
    walks_per_node=EXPLAIN_WALKS_PER_NODE,
    num_negative_samples=EXPLAIN_NEG_SAMPLES,
    batch_size=128,
    lr=0.01,
):
    num_nodes = data["applicant"].num_nodes

    mp2v_model = MetaPath2Vec(
        data.edge_index_dict,
        embedding_dim=emb_dim,
        metapath=metapath,
        walk_length=walk_length,
        context_size=context_size,
        walks_per_node=walks_per_node,
        num_negative_samples=num_negative_samples,
        sparse=True
    ).to(device)

    optimizer = torch.optim.SparseAdam(mp2v_model.parameters(), lr=lr)
    loader = mp2v_model.loader(batch_size=batch_size, shuffle=True)

    stored_pos_rw = []
    loss_history = []

    mp2v_model.train()
    for epoch in range(epochs):
        batch_losses = []

        for pos_rw, neg_rw in loader:
            pos_rw = pos_rw.to(device)
            neg_rw = neg_rw.to(device)

            # store positive training walk windows
            stored_pos_rw.append(pos_rw.detach().cpu())

            optimizer.zero_grad()
            loss = mp2v_model.loss(pos_rw, neg_rw)

            if torch.isnan(loss):
                continue

            loss.backward()
            optimizer.step()
            batch_losses.append(float(loss.item()))

        loss_history.append(sum(batch_losses) / max(1, len(batch_losses)))

    mp2v_model.eval()
    with torch.no_grad():
        emb = mp2v_model("applicant").detach().cpu()

    emb = torch.nan_to_num(emb, nan=0.0, posinf=5.0, neginf=-5.0)

    if emb.shape[0] != num_nodes:
        full_emb = torch.zeros(num_nodes, emb_dim)
        full_emb[:emb.shape[0]] = emb
        emb = full_emb

    pos_rw_all = torch.cat(stored_pos_rw, dim=0) if stored_pos_rw else torch.empty((0, context_size), dtype=torch.long)

    return {
        "embedding": emb,
        "model": mp2v_model,
        "pos_rw": pos_rw_all,
        "loss_history": loss_history,
        "metapath": metapath,
        "num_nodes": num_nodes,
    }

def build_explainer_cache_for_year(year, metapath_names=None):
    if metapath_names is None:
        metapath_names = list(METAPATHS.keys())

    cache = {}
    data = data_by_year[year]

    for meta_name in metapath_names:
        print(f"Training explainable clone for year={year}, metapath={meta_name}")
        cache[meta_name] = train_metapath_embedding_with_walk_windows(
            data=data,
            metapath=METAPATHS[meta_name]["path"]
        )

    return cache

In [9]:
# ============================================================
# CELL C — BUILD EXPLAINER CACHE FOR ONE OR MORE METAPATHS
# Start with one metapath to keep it light
# ============================================================

# Example:
# explain_cache = build_explainer_cache_for_year(EXPLAIN_YEAR, metapath_names=["APC"])

# If you want all three:
explain_cache = build_explainer_cache_for_year(
    EXPLAIN_YEAR,
    metapath_names=["APA", "API", "APC"]
)

print("Built explain_cache for:", list(explain_cache.keys()))
for k, v in explain_cache.items():
    print(k, v["embedding"].shape, v["pos_rw"].shape)

Training explainable clone for year=2024, metapath=APA
Training explainable clone for year=2024, metapath=API
Training explainable clone for year=2024, metapath=APC
Built explain_cache for: ['APA', 'API', 'APC']
APA torch.Size([4400, 32]) torch.Size([697896, 5])
API torch.Size([4400, 32]) torch.Size([697896, 5])
APC torch.Size([4400, 32]) torch.Size([697896, 5])


In [10]:
# ============================================================
# CELL D — HELPERS TO SCORE PAIRS WITH YOUR FINAL TEMPORAL MODEL
# Reuses the already-trained temporal model + z
# ============================================================

def predict_pair_probability(app_u, app_v, temporal_model=None, temporal_z=None):
    temporal_model = model if temporal_model is None else temporal_model
    temporal_z = z if temporal_z is None else temporal_z

    u = resolve_applicant_id(app_u)
    v = resolve_applicant_id(app_v)

    edge_idx = torch.tensor([[u], [v]], dtype=torch.long, device=device)
    temporal_model.eval()
    with torch.no_grad():
        z_dev = temporal_z.to(device) if temporal_z.device.type == "cpu" else temporal_z
        proba = temporal_model.decode_proba(z_dev, edge_idx).item()

    return {
        "u_id": u,
        "v_id": v,
        "u_name": id_to_applicant[u],
        "v_name": id_to_applicant[v],
        "probability": float(proba),
        "is_train_edge": (min(u, v), max(u, v)) in TRAIN_EDGE_SET,
        "is_test_edge": (min(u, v), max(u, v)) in TEST_EDGE_SET,
    }

def top_k_predictions_for_applicant(applicant_name, k=10, exclude_train_edges=True, exclude_self=True):
    u = resolve_applicant_id(applicant_name)
    z_dev = z.to(device) if z.device.type == "cpu" else z

    candidates = []
    for v in range(z_dev.size(0)):
        if exclude_self and v == u:
            continue
        if exclude_train_edges and (min(u, v), max(u, v)) in TRAIN_EDGE_SET:
            continue
        candidates.append(v)

    if not candidates:
        return pd.DataFrame()

    src = torch.full((len(candidates),), u, dtype=torch.long)
    dst = torch.tensor(candidates, dtype=torch.long)
    edge_idx = torch.stack([src, dst], dim=0).to(device)

    model.eval()
    with torch.no_grad():
        probs = model.decode_proba(z_dev, edge_idx).detach().cpu().numpy()

    df_out = pd.DataFrame({
        "source_id": u,
        "source_name": id_to_applicant[u],
        "target_id": candidates,
        "target_name": [id_to_applicant[v] for v in candidates],
        "probability": probs,
        "is_test_edge": [(min(u, v), max(u, v)) in TEST_EDGE_SET for v in candidates],
    }).sort_values("probability", ascending=False).head(k).reset_index(drop=True)

    return df_out

# Example:
# top_k_predictions_for_applicant("your applicant here", k=5)

In [11]:
# ============================================================
# CELL E — WALK-WINDOW EXPLAINER FOR A PAIR, WITHIN ONE METAPATH
# ============================================================

def explain_pair_within_metapath(
    app_u,
    app_v,
    explain_run,
    top_k=10,
    require_touch=True,
    aggregate_duplicates=True
):
    u = resolve_applicant_id(app_u)
    v = resolve_applicant_id(app_v)

    mp2v_model = explain_run["model"]
    pos_rw = explain_run["pos_rw"]
    app_emb = explain_run["embedding"]

    if pos_rw.numel() == 0:
        return pd.DataFrame()

    type_emb_cache = build_type_embedding_cache(mp2v_model)

    emb_u = app_emb[u]
    emb_v = app_emb[v]
    pair_anchor = F.normalize((emb_u + emb_v).view(1, -1), dim=1).view(-1)

    rows = pos_rw
    if aggregate_duplicates:
        rows, counts = torch.unique(rows, dim=0, return_counts=True)
    else:
        counts = torch.ones(rows.size(0), dtype=torch.long)

    results = []

    for i in range(rows.size(0)):
        rw = rows[i]
        freq = int(counts[i].item())
        decoded = decode_rw_row(rw, mp2v_model)

        applicant_ids_in_rw = [
            item["local_id"]
            for item in decoded
            if item["node_type"] == "applicant" and item["local_id"] is not None
        ]

        if require_touch and (u not in applicant_ids_in_rw and v not in applicant_ids_in_rw):
            continue

        wvec = walk_window_embedding(decoded, type_emb_cache)
        if wvec is None:
            continue

        support_u = cosine_safe(wvec, emb_u)
        support_v = cosine_safe(wvec, emb_v)
        support_pair = cosine_safe(wvec, pair_anchor)

        has_u = u in applicant_ids_in_rw
        has_v = v in applicant_ids_in_rw
        bridge_bonus = 0.50 if (has_u and has_v) else 0.0

        # emphasize windows that actually touch one of the focal applicants
        touch_bonus = 0.15 if (has_u or has_v) else 0.0

        # frequency matters because repeated windows affected training more often
        weighted_influence = (0.30 * support_u + 0.30 * support_v + 0.40 * support_pair + bridge_bonus + touch_bonus)
        weighted_influence = weighted_influence * math.log1p(freq)

        results.append({
            "frequency": freq,
            "support_u": support_u,
            "support_v": support_v,
            "support_pair": support_pair,
            "bridge_bonus": bridge_bonus,
            "weighted_influence": weighted_influence,
            "contains_u": has_u,
            "contains_v": has_v,
            "decoded_walk_window": format_decoded_rw(decoded),
            "raw_tokens": rw.tolist(),
        })

    if not results:
        return pd.DataFrame()

    return pd.DataFrame(results).sort_values(
        ["weighted_influence", "frequency"],
        ascending=False
    ).head(top_k).reset_index(drop=True)

In [12]:
# ============================================================
# CELL F — COMPARE METAPATH SUPPORT FOR THE SAME PAIR
# ============================================================

def compare_metapath_support_for_pair(app_u, app_v, explain_cache, top_k_per_meta=10):
    summaries = []

    for meta_name, explain_run in explain_cache.items():
        df_meta = explain_pair_within_metapath(
            app_u=app_u,
            app_v=app_v,
            explain_run=explain_run,
            top_k=top_k_per_meta,
            require_touch=True,
            aggregate_duplicates=True
        )

        if df_meta.empty:
            summaries.append({
                "metapath": meta_name,
                "num_supporting_windows": 0,
                "top_window_influence": 0.0,
                "avg_top_window_influence": 0.0,
            })
            continue

        summaries.append({
            "metapath": meta_name,
            "num_supporting_windows": int(len(df_meta)),
            "top_window_influence": float(df_meta["weighted_influence"].max()),
            "avg_top_window_influence": float(df_meta["weighted_influence"].mean()),
        })

    return pd.DataFrame(summaries).sort_values(
        ["top_window_influence", "avg_top_window_influence"],
        ascending=False
    ).reset_index(drop=True)

In [27]:
# ============================================================
# CELL G — BUILD EXPLANATION PAIRS FROM ALL top_links.json PAIRS
# ============================================================

import os
import json
import pandas as pd

EXPLAIN_EXPORT_DIR = os.path.join(DATA_DIR, "predictions", "explanations")
os.makedirs(EXPLAIN_EXPORT_DIR, exist_ok=True)

TOP_LINKS_PATH = os.path.join(DATA_DIR, "predictions", "top_links.json")

def build_pairs_from_top_links_json(top_links_path, keep_only_predicted=True):
    with open(top_links_path, "r") as f:
        top_links_data = json.load(f)

    pair_map = {}

    for source_key, link_list in top_links_data.items():
        source_id = int(source_key)

        for item in link_list:
            target_id = int(item["target"])
            score = float(item["score"])

            if source_id == target_id:
                continue

            a, b = (source_id, target_id) if source_id < target_id else (target_id, source_id)
            pair_id = f"{a}__{b}"

            status = "Predicted"
            if (a, b) in TRAIN_EDGE_SET:
                status = "Historical"
            elif (a, b) in TEST_EDGE_SET:
                status = "Observed in test year"

            if keep_only_predicted and status != "Predicted":
                continue

            if pair_id not in pair_map or score > pair_map[pair_id]["prediction_score"]:
                pair_map[pair_id] = {
                    "pair_id": pair_id,
                    "source_id": a,
                    "source_name": id_to_applicant[a],
                    "target_id": b,
                    "target_name": id_to_applicant[b],
                    "prediction_score": score,
                    "status": status
                }

    df_pairs = pd.DataFrame(pair_map.values())
    if df_pairs.empty:
        return df_pairs

    df_pairs = df_pairs.sort_values("prediction_score", ascending=False).reset_index(drop=True)
    df_pairs["rank"] = range(1, len(df_pairs) + 1)

    cols = [
        "rank", "pair_id",
        "source_id", "source_name",
        "target_id", "target_name",
        "prediction_score", "status"
    ]
    return df_pairs[cols]

explanation_pairs_df = build_pairs_from_top_links_json(
    TOP_LINKS_PATH,
    keep_only_predicted=True
)

display(explanation_pairs_df.head(10))
print("Total explanation pairs:", len(explanation_pairs_df))

explanation_pairs_df.to_csv(
    os.path.join(EXPLAIN_EXPORT_DIR, "explanation_pairs.csv"),
    index=False
)

,rank,pair_id,source_id,source_name,target_id,target_name,prediction_score,status
0,1,1728__3046,1728,hoffmann la roche,3046,qiagen gmbh,0.999665,Predicted
1,2,814__2180,814,consejo superior investigacion,2180,ladner robert c,0.999663,Predicted
2,3,1728__4002,1728,hoffmann la roche,4002,univ syddansk,0.999629,Predicted
3,4,611__2880,611,cantargia ab,2880,pasteur institut,0.999593,Predicted
4,5,1737__3765,1737,holm per sonne,3765,turun yliopisto,0.999576,Predicted
5,6,1728__2434,1728,hoffmann la roche,2434,massachusetts inst technology,0.999539,Predicted
6,7,817__2593,817,consiglio nazionale ricerche,2593,mologen ag,0.999522,Predicted
7,8,353__1871,353,bernards rene,1871,inst nat sante rech med,0.999499,Predicted
8,9,1310__2880,1310,fundacio inst d investigacio biomedica de bell...,2880,pasteur institut,0.999458,Predicted
9,10,293__3282,293,bayer ag,3282,santaris pharma as,0.999455,Predicted


Total explanation pairs: 212042


In [29]:
# ============================================================
# CELL H — BUILD EXPLANATION TABLES FOR THE GLOBAL TOP 10
# Add after Cell G
# ============================================================

import time
import json
import pandas as pd

def build_explanation_exports_for_pairs(
    pairs_df,
    explain_cache,
    top_walks_per_metapath=5,
    top_metapaths_per_pair=None
):
    pair_summary_rows = []
    metapath_rows = []
    walk_rows = []

    total_pairs = len(pairs_df)
    overall_start = time.time()

    for pair_idx, (_, pair) in enumerate(pairs_df.iterrows(), start=1):
        pair_start = time.time()

        app_u = pair["source_name"]
        app_v = pair["target_name"]
        pair_id = pair["pair_id"]

        print(f"\n[Pair {pair_idx}/{total_pairs}] {app_u} -> {app_v} (rank={pair['rank']})")

        meta_summary = compare_metapath_support_for_pair(
            app_u,
            app_v,
            explain_cache,
            top_k_per_meta=max(10, top_walks_per_metapath)
        )

        print(f"  meta_summary done | rows={len(meta_summary)} | elapsed={time.time() - pair_start:.2f}s")

        if meta_summary.empty:
            pair_summary_rows.append({
                "pair_id": pair_id,
                "rank": int(pair["rank"]),
                "source_id": int(pair["source_id"]),
                "source_name": app_u,
                "target_id": int(pair["target_id"]),
                "target_name": app_v,
                "prediction_score": float(pair["prediction_score"]),
                "best_metapath": None,
                "best_metapath_top_window_influence": None,
                "best_metapath_avg_top_window_influence": None,
                "num_metapaths_with_support": 0
            })
            print(f"  no metapath support found | pair elapsed={time.time() - pair_start:.2f}s")
            continue

        if top_metapaths_per_pair is not None:
            meta_summary = meta_summary.head(top_metapaths_per_pair).copy()

        best_meta = meta_summary.iloc[0]

        pair_summary_rows.append({
            "pair_id": pair_id,
            "rank": int(pair["rank"]),
            "source_id": int(pair["source_id"]),
            "source_name": app_u,
            "target_id": int(pair["target_id"]),
            "target_name": app_v,
            "prediction_score": float(pair["prediction_score"]),
            "best_metapath": best_meta["metapath"],
            "best_metapath_top_window_influence": float(best_meta["top_window_influence"]),
            "best_metapath_avg_top_window_influence": float(best_meta["avg_top_window_influence"]),
            "num_metapaths_with_support": int((meta_summary["num_supporting_windows"] > 0).sum())
        })

        for meta_rank, (_, meta_row) in enumerate(meta_summary.iterrows(), start=1):
            meta_start = time.time()
            metapath_name = meta_row["metapath"]

            print(f"    [Metapath {meta_rank}/{len(meta_summary)}] {metapath_name}")

            metapath_rows.append({
                "pair_id": pair_id,
                "pair_rank": int(pair["rank"]),
                "metapath_rank": int(meta_rank),
                "metapath": metapath_name,
                "num_supporting_windows": int(meta_row["num_supporting_windows"]),
                "top_window_influence": float(meta_row["top_window_influence"]),
                "avg_top_window_influence": float(meta_row["avg_top_window_influence"]),
            })

            walk_df = explain_pair_within_metapath(
                app_u,
                app_v,
                explain_cache[metapath_name],
                top_k=top_walks_per_metapath,
                require_touch=True,
                aggregate_duplicates=True
            )

            print(f"      walk_df done | rows={len(walk_df)} | elapsed={time.time() - meta_start:.2f}s")

            if walk_df.empty:
                continue

            for walk_rank, (_, walk_row) in enumerate(walk_df.iterrows(), start=1):
                walk_rows.append({
                    "pair_id": pair_id,
                    "pair_rank": int(pair["rank"]),
                    "metapath": metapath_name,
                    "walk_rank": int(walk_rank),
                    "frequency": int(walk_row["frequency"]),
                    "weighted_influence": float(walk_row["weighted_influence"]),
                    "support_u": float(walk_row["support_u"]),
                    "support_v": float(walk_row["support_v"]),
                    "support_pair": float(walk_row["support_pair"]),
                    "contains_u": bool(walk_row["contains_u"]),
                    "contains_v": bool(walk_row["contains_v"]),
                    "decoded_walk_window": str(walk_row["decoded_walk_window"]),
                    "raw_tokens_json": json.dumps(walk_row["raw_tokens"]),
                })

        print(f"  pair finished | elapsed={time.time() - pair_start:.2f}s")

    pair_summary_df = pd.DataFrame(pair_summary_rows)
    metapath_df = pd.DataFrame(metapath_rows)
    walk_df = pd.DataFrame(walk_rows)

    print(f"\nDone. Total elapsed: {time.time() - overall_start:.2f}s")
    print(f"pair_summary_df shape: {pair_summary_df.shape}")
    print(f"metapath_df shape: {metapath_df.shape}")
    print(f"walk_df shape: {walk_df.shape}")

    return pair_summary_df, metapath_df, walk_df


# -----------------------------------
# Reduce to TOP 10 pairs only
# -----------------------------------
top10_explanation_pairs_df = explanation_pairs_df.head(10).copy()

pair_summary_df, metapath_explanations_df, walk_explanations_df = build_explanation_exports_for_pairs(
    pairs_df=top10_explanation_pairs_df,
    explain_cache=explain_cache,
    top_walks_per_metapath=5,
    top_metapaths_per_pair=3
)

display(pair_summary_df.head())
display(metapath_explanations_df.head())
display(walk_explanations_df.head())


[Pair 1/10] hoffmann la roche -> qiagen gmbh (rank=1)
  meta_summary done | rows=3 | elapsed=4.50s
    [Metapath 1/3] API
      walk_df done | rows=2 | elapsed=1.13s
    [Metapath 2/3] APA
      walk_df done | rows=2 | elapsed=1.07s
    [Metapath 3/3] APC
      walk_df done | rows=2 | elapsed=1.16s
  pair finished | elapsed=7.86s

[Pair 2/10] consejo superior investigacion -> ladner robert c (rank=2)
  meta_summary done | rows=3 | elapsed=3.36s
    [Metapath 1/3] API
      walk_df done | rows=2 | elapsed=1.10s
    [Metapath 2/3] APA
      walk_df done | rows=2 | elapsed=1.07s
    [Metapath 3/3] APC
      walk_df done | rows=2 | elapsed=1.10s
  pair finished | elapsed=6.63s

[Pair 3/10] hoffmann la roche -> univ syddansk (rank=3)
  meta_summary done | rows=3 | elapsed=3.33s
    [Metapath 1/3] APC
      walk_df done | rows=5 | elapsed=1.11s
    [Metapath 2/3] APA
      walk_df done | rows=5 | elapsed=1.07s
    [Metapath 3/3] API
      walk_df done | rows=5 | elapsed=1.11s
  pair finishe

,pair_id,rank,source_id,source_name,target_id,target_name,prediction_score,best_metapath,best_metapath_top_window_influence,best_metapath_avg_top_window_influence,num_metapaths_with_support
0,1728__3046,1,1728,hoffmann la roche,3046,qiagen gmbh,0.999665,API,1.531932,1.480169,3
1,814__2180,2,814,consejo superior investigacion,2180,ladner robert c,0.999663,API,1.520471,1.511777,3
2,1728__4002,3,1728,hoffmann la roche,4002,univ syddansk,0.999629,APC,1.645574,0.518406,3
3,611__2880,4,611,cantargia ab,2880,pasteur institut,0.999593,API,1.746746,0.727582,3
4,1737__3765,5,1737,holm per sonne,3765,turun yliopisto,0.999576,APA,1.762520,1.737339,3


,pair_id,pair_rank,metapath_rank,metapath,num_supporting_windows,top_window_influence,avg_top_window_influence
0,1728__3046,1,1,API,2,1.531932,1.480169
1,1728__3046,1,2,APA,2,1.527645,1.457929
2,1728__3046,1,3,APC,2,1.363664,1.312993
3,814__2180,2,1,API,2,1.520471,1.511777
4,814__2180,2,2,APA,2,1.471936,1.403045


,pair_id,pair_rank,metapath,walk_rank,frequency,weighted_influence,support_u,support_v,support_pair,contains_u,contains_v,decoded_walk_window,raw_tokens_json
0,1728__3046,1,API,1,6,1.531932,0.066494,1.000000,0.793272,False,True,applicant:qiagen gmbh -> <dummy> -> <dummy> ->...,"[3046, 17621, 17621, 17621, 17621]"
1,1728__3046,1,API,2,6,1.428405,1.000000,0.066494,0.660267,True,False,applicant:hoffmann la roche -> <dummy> -> <dum...,"[1728, 17621, 17621, 17621, 17621]"
2,1728__3046,1,APA,1,6,1.527645,1.000000,0.042768,0.805560,True,False,applicant:hoffmann la roche -> <dummy> -> <dum...,"[1728, 9667, 9667, 9667, 9667]"
3,1728__3046,1,APA,2,6,1.388213,0.042768,1.000000,0.626425,False,True,applicant:qiagen gmbh -> <dummy> -> <dummy> ->...,"[3046, 9667, 9667, 9667, 9667]"
4,1728__3046,1,APC,1,6,1.363664,-0.127960,1.000000,0.722932,False,True,applicant:qiagen gmbh -> <dummy> -> <dummy> ->...,"[3046, 13205, 13205, 13205, 13205]"


In [30]:
walk_explanations_df['decoded_walk_window']

0      applicant:qiagen gmbh -> <dummy> -> <dummy> ->...
1      applicant:hoffmann la roche -> <dummy> -> <dum...
2      applicant:hoffmann la roche -> <dummy> -> <dum...
3      applicant:qiagen gmbh -> <dummy> -> <dummy> ->...
4      applicant:qiagen gmbh -> <dummy> -> <dummy> ->...
                             ...                        
100    applicant:santaris pharma as -> <dummy> -> <du...
101    applicant:bayer ag -> patent:US 2021/0054979 W...
102    applicant:bayer ag -> patent:US 202063092973 P...
103    applicant:bayer ag -> patent:US 202118032096 A...
104    applicant: -> patent:US 202118032096 A -> appl...
Name: decoded_walk_window, Length: 105, dtype: object

In [34]:
# ============================================================
# CELL I — SAVE DASHBOARD-READY FILES
# Add after Cell H
# ============================================================

pair_summary_path = os.path.join(EXPLAIN_EXPORT_DIR, "pair_summary.csv")
metapath_path     = os.path.join(EXPLAIN_EXPORT_DIR, "metapath_explanations.csv")
walks_path        = os.path.join(EXPLAIN_EXPORT_DIR, "walk_explanations.csv")
manifest_path     = os.path.join(EXPLAIN_EXPORT_DIR, "explanations_manifest.json")

pair_summary_df.to_csv(pair_summary_path, index=False)
metapath_explanations_df.to_csv(metapath_path, index=False)
walk_explanations_df.to_csv(walks_path, index=False)

manifest = {
    "top_k_pairs": int(len(pair_summary_df)),
    "files": {
        "pair_summary": pair_summary_path,
        "metapath_explanations": metapath_path,
        "walk_explanations": walks_path,
        "top_pairs_input": os.path.join(EXPLAIN_EXPORT_DIR, "top_50_pairs.csv"),
    },
    "columns": {
        "pair_summary": list(pair_summary_df.columns),
        "metapath_explanations": list(metapath_explanations_df.columns),
        "walk_explanations": list(walk_explanations_df.columns),
    }
}

with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2)

print("Saved dashboard-ready explanation files:")
print("-", pair_summary_path)
print("-", metapath_path)
print("-", walks_path)
print("-", manifest_path)

Saved dashboard-ready explanation files:
- /Users/carlos38/Desktop/Profesional/Master/EU_collabo_project/data/processed/predictions/explanations/pair_summary.csv
- /Users/carlos38/Desktop/Profesional/Master/EU_collabo_project/data/processed/predictions/explanations/metapath_explanations.csv
- /Users/carlos38/Desktop/Profesional/Master/EU_collabo_project/data/processed/predictions/explanations/walk_explanations.csv
- /Users/carlos38/Desktop/Profesional/Master/EU_collabo_project/data/processed/predictions/explanations/explanations_manifest.json


In [35]:
# ============================================================
# CELL J — OPTIONAL NESTED JSON FOR STREAMLIT
# Add after Cell I
# ============================================================

nested_json_path = os.path.join(EXPLAIN_EXPORT_DIR, "explanations_nested.json")

nested = []

for _, pair_row in pair_summary_df.iterrows():
    pair_id = pair_row["pair_id"]

    meta_rows = metapath_explanations_df[
        metapath_explanations_df["pair_id"] == pair_id
    ].copy()

    walk_rows = walk_explanations_df[
        walk_explanations_df["pair_id"] == pair_id
    ].copy()

    metapaths_payload = []
    for _, mrow in meta_rows.iterrows():
        mp = mrow["metapath"]

        mp_walks = walk_rows[walk_rows["metapath"] == mp].copy()

        metapaths_payload.append({
            "metapath": mp,
            "metapath_rank": int(mrow["metapath_rank"]),
            "num_supporting_windows": int(mrow["num_supporting_windows"]),
            "top_window_influence": float(mrow["top_window_influence"]),
            "avg_top_window_influence": float(mrow["avg_top_window_influence"]),
            "top_walks": [
                {
                    "walk_rank": int(w["walk_rank"]),
                    "frequency": int(w["frequency"]),
                    "weighted_influence": float(w["weighted_influence"]),
                    "support_u": float(w["support_u"]),
                    "support_v": float(w["support_v"]),
                    "support_pair": float(w["support_pair"]),
                    "decoded_walk_window": str(w["decoded_walk_window"]),
                    "raw_tokens": json.loads(w["raw_tokens_json"]),
                }
                for _, w in mp_walks.sort_values("walk_rank").iterrows()
            ]
        })

    nested.append({
        "pair_id": pair_id,
        "rank": int(pair_row["rank"]),
        "source_id": int(pair_row["source_id"]),
        "source_name": pair_row["source_name"],
        "target_id": int(pair_row["target_id"]),
        "target_name": pair_row["target_name"],
        "prediction_score": float(pair_row["prediction_score"]),
        "best_metapath": pair_row["best_metapath"],
        "best_metapath_top_window_influence": (
            None if pd.isna(pair_row["best_metapath_top_window_influence"])
            else float(pair_row["best_metapath_top_window_influence"])
        ),
        "metapaths": metapaths_payload
    })

with open(nested_json_path, "w") as f:
    json.dump(nested, f, indent=2)

print("Saved nested explanation JSON:")
print("-", nested_json_path)

Saved nested explanation JSON:
- /Users/carlos38/Desktop/Profesional/Master/EU_collabo_project/data/processed/predictions/explanations/explanations_nested.json


In [24]:
pair_id = "123__456"   # clicked pair
pair_summary_df[pair_summary_df["pair_id"] == pair_id]

,pair_id,rank,source_id,source_name,target_id,target_name,prediction_score,best_metapath,best_metapath_top_window_influence,best_metapath_avg_top_window_influence,num_metapaths_with_support


In [40]:
import pandas as pd
import numpy as np
import re

# =========================================================
# CELL I — POST-PROCESS WALK EXPLANATIONS FROM CELL H
# Coherent with Cell H outputs:
#   pair_summary_df
#   walk_explanations_df (or exported walk explanations table)
# =========================================================

# =========================================================
# 1. INPUTS
# =========================================================
# Expected from Cell H:
#   pair_summary_df: pair-level summary table
#   walk_explanations_df: walk-level explanation table
#
# Required columns in pair_summary_df:
#   pair_id, rank, source_id, source_name, target_id, target_name
#
# Required columns in walk_explanations_df:
#   pair_id, metapath, walk_rank, frequency, weighted_influence,
#   support_u, support_v, support_pair, decoded_walk_window

pair_lookup_df = pair_summary_df.copy()
explanations_df = walk_explanations_df.copy()


# =========================================================
# 2. BUILD LOOKUP DICTIONARY
# =========================================================
pair_lookup = (
    pair_lookup_df[
        ["pair_id", "rank", "source_id", "source_name", "target_id", "target_name"]
    ]
    .drop_duplicates(subset=["pair_id"])
    .set_index("pair_id")
    .to_dict(orient="index")
)


# =========================================================
# 3. HELPERS
# =========================================================
def safe_float(x, default=0.0):
    try:
        if pd.isna(x):
            return default
        return float(x)
    except Exception:
        return default


def normalize_token(token):
    if pd.isna(token):
        return ""
    return str(token).strip().lower()


def split_walk(decoded_walk_window):
    """
    Supports both:
      applicant:a → patent:x → applicant:b
    and
      applicant:a -> patent:x -> applicant:b
    """
    if pd.isna(decoded_walk_window):
        return []
    parts = re.split(r"\s*(?:→|->)\s*", str(decoded_walk_window).strip())
    return [x.strip() for x in parts if x.strip()]


def is_dummy_token(token):
    tok = normalize_token(token)
    return tok in {"<dummy>", "dummy", "[dummy]"}


def is_applicant_token(token):
    return normalize_token(token).startswith("applicant:")


def is_inventor_token(token):
    return normalize_token(token).startswith("inventor:")


def is_cpc_token(token):
    return normalize_token(token).startswith("cpc:")


def strip_prefix(token):
    tok = normalize_token(token)
    if ":" in tok:
        return tok.split(":", 1)[1].strip()
    return tok


def clean_entity_label(token):
    """
    Converts:
      applicant:hoffmann la roche -> hoffmann la roche
      inventor:john doe -> john doe
      cpc:a61k 39/00 -> a61k 39/00
    """
    return strip_prefix(token)


def get_pair_nodes(pair_id, pair_lookup):
    """
    Return source/target variants so we can correctly exclude
    focal applicants whether they appear as raw text or applicant-prefixed.
    """
    entry = pair_lookup.get(pair_id, {})
    source_name = normalize_token(entry.get("source_name", ""))
    target_name = normalize_token(entry.get("target_name", ""))

    source_tokens = {
        source_name,
        f"applicant:{source_name}"
    }
    target_tokens = {
        target_name,
        f"applicant:{target_name}"
    }
    return source_tokens, target_tokens


def get_pair_metadata(pair_id, pair_lookup):
    entry = pair_lookup.get(pair_id, {})
    return {
        "rank": entry.get("rank"),
        "source_id": entry.get("source_id"),
        "source_name": entry.get("source_name"),
        "target_id": entry.get("target_id"),
        "target_name": entry.get("target_name"),
    }


# =========================================================
# 4. CLASSIFY WALKS AND EXTRACT INFORMATIVE ENTITIES
# =========================================================
def classify_and_extract_walk(row, pair_lookup):
    """
    Classifies each walk into:
      - endpoint_only
      - partially_interpretable
      - interpretable

    Extracts informative intermediate entities depending on metapath:
      API -> inventors
      APA -> third applicants
      APC -> CPCs

    Notes:
    - excludes focal source and target applicants
    - ignores dummy tokens
    - deduplicates repeated entities within the same walk
    """
    pair_id = row["pair_id"]
    metapath = row["metapath"]
    decoded = row.get("decoded_walk_window", "")

    source_tokens, target_tokens = get_pair_nodes(pair_id, pair_lookup)
    tokens = split_walk(decoded)

    informative_entities = []
    seen = set()

    for tok in tokens:
        tok_norm = normalize_token(tok)

        if not tok_norm:
            continue
        if is_dummy_token(tok_norm):
            continue
        if tok_norm in source_tokens or tok_norm in target_tokens:
            continue

        entity_type = None

        if metapath == "API" and is_inventor_token(tok_norm):
            entity_type = "inventor"
        elif metapath == "APA" and is_applicant_token(tok_norm):
            entity_type = "third_applicant"
        elif metapath == "APC" and is_cpc_token(tok_norm):
            entity_type = "cpc"

        if entity_type is not None:
            key = (tok_norm, entity_type)
            if key in seen:
                continue
            seen.add(key)

            informative_entities.append({
                "token": tok_norm,
                "clean_label": clean_entity_label(tok_norm),
                "entity_type": entity_type
            })

    n_entities = len(informative_entities)

    if n_entities == 0:
        walk_class = "endpoint_only"
    elif n_entities == 1:
        walk_class = "partially_interpretable"
    else:
        walk_class = "interpretable"

    return walk_class, informative_entities


# =========================================================
# 5. AGGREGATE WALK- AND ENTITY-LEVEL EVIDENCE
# =========================================================
def aggregate_explanations(explanations_df, pair_lookup):
    walk_rows = []
    entity_rows = []

    for _, row in explanations_df.iterrows():
        pair_id = row["pair_id"]
        meta = get_pair_metadata(pair_id, pair_lookup)

        walk_class, entities = classify_and_extract_walk(row, pair_lookup)

        weighted_influence = safe_float(row.get("weighted_influence", 0))
        support_pair = safe_float(row.get("support_pair", 0))
        support_u = safe_float(row.get("support_u", 0))
        support_v = safe_float(row.get("support_v", 0))
        frequency = safe_float(row.get("frequency", 1))

        walk_score = weighted_influence * max(0.0, support_pair)
        balance_factor = max(0.0, 1.0 - abs(support_u - support_v))
        balanced_walk_score = walk_score * balance_factor

        walk_rows.append({
            "pair_id": pair_id,
            "rank": meta["rank"],
            "source_id": meta["source_id"],
            "source_name": meta["source_name"],
            "target_id": meta["target_id"],
            "target_name": meta["target_name"],
            "metapath": row["metapath"],
            "walk_rank": row.get("walk_rank"),
            "walk_class": walk_class,
            "decoded_walk_window": row.get("decoded_walk_window"),
            "frequency": frequency,
            "weighted_influence": weighted_influence,
            "support_u": support_u,
            "support_v": support_v,
            "support_pair": support_pair,
            "walk_score": walk_score,
            "balanced_walk_score": balanced_walk_score,
            "n_informative_entities": len(entities)
        })

        if walk_class == "endpoint_only" or len(entities) == 0:
            continue

        n_entities = len(entities)
        entity_score = walk_score / n_entities
        balanced_entity_score = balanced_walk_score / n_entities

        for ent in entities:
            entity_rows.append({
                "pair_id": pair_id,
                "rank": meta["rank"],
                "source_name": meta["source_name"],
                "target_name": meta["target_name"],
                "metapath": row["metapath"],
                "entity_type": ent["entity_type"],
                "entity": ent["token"],
                "entity_label": ent["clean_label"],
                "walk_class": walk_class,
                "weighted_influence": weighted_influence,
                "support_pair": support_pair,
                "support_u": support_u,
                "support_v": support_v,
                "frequency": frequency,
                "n_entities_in_walk": n_entities,
                "entity_score": entity_score,
                "balanced_entity_score": balanced_entity_score
            })

    walk_df = pd.DataFrame(walk_rows)
    entity_df = pd.DataFrame(entity_rows)

    return walk_df, entity_df


walk_df, entity_df = aggregate_explanations(explanations_df, pair_lookup)


# =========================================================
# 6. PAIR-LEVEL EXPLANATION QUALITY
# =========================================================
def explanation_quality_by_pair(walk_df):
    tmp = (
        walk_df.groupby(["pair_id", "walk_class"], dropna=False)
        .agg(total_support=("walk_score", "sum"))
        .reset_index()
    )

    total = (
        tmp.groupby("pair_id", dropna=False)["total_support"]
        .sum()
        .rename("pair_total_support")
        .reset_index()
    )

    tmp = tmp.merge(total, on="pair_id", how="left")
    tmp["support_share"] = np.where(
        tmp["pair_total_support"] > 0,
        tmp["total_support"] / tmp["pair_total_support"],
        0.0
    )
    return tmp


quality_df = explanation_quality_by_pair(walk_df)


# =========================================================
# 7. METAPATH / WALK-CLASS SUPPORT SUMMARY
# =========================================================
def summarize_support(walk_df):
    return (
        walk_df.groupby(
            ["pair_id", "rank", "source_name", "target_name", "metapath", "walk_class"],
            dropna=False
        )
        .agg(
            total_walks=("walk_score", "count"),
            total_support=("walk_score", "sum"),
            total_balanced_support=("balanced_walk_score", "sum")
        )
        .reset_index()
        .sort_values(["rank", "total_support"], ascending=[True, False])
    )


support_summary_df = summarize_support(walk_df)


# =========================================================
# 8. RANK TOP SUPPORTING ENTITIES
# =========================================================
def rank_entities(entity_df, pair_id, entity_type=None, top_n=10, use_balanced=True):
    score_col = "balanced_entity_score" if use_balanced else "entity_score"

    subset = entity_df[entity_df["pair_id"] == pair_id].copy()
    if entity_type is not None:
        subset = subset[subset["entity_type"] == entity_type]

    if subset.empty:
        return pd.DataFrame(columns=[
            "entity_label", "entity", "entity_type", "score", "occurrences",
            "avg_support_pair", "total_influence"
        ])

    ranked = (
        subset.groupby(["entity_label", "entity", "entity_type"], dropna=False)
        .agg(
            score=(score_col, "sum"),
            occurrences=("entity", "count"),
            avg_support_pair=("support_pair", "mean"),
            total_influence=("weighted_influence", "sum")
        )
        .reset_index()
        .sort_values(["score", "occurrences"], ascending=[False, False])
        .head(top_n)
        .reset_index(drop=True)
    )

    return ranked


# =========================================================
# 9. BUILD A PAIR SUMMARY TABLE
# =========================================================
def build_pair_summary(pair_id, walk_df, entity_df, quality_df):
    pair_walks = walk_df[walk_df["pair_id"] == pair_id].copy()
    pair_quality = quality_df[quality_df["pair_id"] == pair_id].copy()

    if pair_walks.empty:
        return pd.DataFrame()

    meta = pair_walks.iloc[0][
        ["pair_id", "rank", "source_id", "source_name", "target_id", "target_name"]
    ].to_dict()

    endpoint_share = pair_quality.loc[
        pair_quality["walk_class"] == "endpoint_only", "support_share"
    ].sum()

    interpretable_share = pair_quality.loc[
        pair_quality["walk_class"].isin(["partially_interpretable", "interpretable"]),
        "support_share"
    ].sum()

    summary = {
        **meta,
        "n_walks": len(pair_walks),
        "total_walk_support": pair_walks["walk_score"].sum(),
        "total_balanced_walk_support": pair_walks["balanced_walk_score"].sum(),
        "interpretable_support_share": interpretable_share,
        "endpoint_only_support_share": endpoint_share,
        "top_inventors_n": len(rank_entities(entity_df, pair_id, "inventor", top_n=10)),
        "top_cpcs_n": len(rank_entities(entity_df, pair_id, "cpc", top_n=10)),
        "top_third_applicants_n": len(rank_entities(entity_df, pair_id, "third_applicant", top_n=10)),
    }

    return pd.DataFrame([summary])


# =========================================================
# 10. OPTIONAL HUMAN-READABLE LABELS
# =========================================================
METAPATH_LABELS = {
    "API": "Inventor-based evidence",
    "APA": "Related applicant evidence",
    "APC": "Technology-class evidence"
}

WALK_CLASS_LABELS = {
    "endpoint_only": "Endpoint-only support",
    "partially_interpretable": "Partial evidence",
    "interpretable": "Interpretable evidence"
}


# =========================================================
# 11. EXAMPLE USAGE — USE TOP-RANKED PAIR FROM CELL H
# =========================================================
example_pair_id = (
    pair_summary_df
    .sort_values("rank", ascending=True)["pair_id"]
    .iloc[0]
)

pair_summary_one_df = build_pair_summary(example_pair_id, walk_df, entity_df, quality_df)
top_inventors_df = rank_entities(entity_df, example_pair_id, "inventor", top_n=10)
top_cpcs_df = rank_entities(entity_df, example_pair_id, "cpc", top_n=10)
top_third_applicants_df = rank_entities(entity_df, example_pair_id, "third_applicant", top_n=10)

print("PAIR SUMMARY")
display(pair_summary_one_df)

print("\nTOP INVENTORS")
display(top_inventors_df)

print("\nTOP CPCS")
display(top_cpcs_df)

print("\nTOP THIRD APPLICANTS")
display(top_third_applicants_df)

print("\nWALK-LEVEL TECHNICAL DETAILS")
display(
    walk_df[walk_df["pair_id"] == example_pair_id]
    .sort_values(["walk_score", "walk_rank"], ascending=[False, True])
    .reset_index(drop=True)
)

PAIR SUMMARY


,pair_id,rank,source_id,source_name,target_id,target_name,n_walks,total_walk_support,total_balanced_walk_support,interpretable_support_share,endpoint_only_support_share,top_inventors_n,top_cpcs_n,top_third_applicants_n
0,1728__3046,1,1728,hoffmann la roche,3046,qiagen gmbh,6,5.992646,0.233341,0.0,1.0,0,0,0



TOP INVENTORS


,entity_label,entity,entity_type,score,occurrences,avg_support_pair,total_influence



TOP CPCS


,entity_label,entity,entity_type,score,occurrences,avg_support_pair,total_influence



TOP THIRD APPLICANTS


,entity_label,entity,entity_type,score,occurrences,avg_support_pair,total_influence



WALK-LEVEL TECHNICAL DETAILS


,pair_id,rank,source_id,source_name,target_id,target_name,metapath,walk_rank,walk_class,decoded_walk_window,frequency,weighted_influence,support_u,support_v,support_pair,walk_score,balanced_walk_score,n_informative_entities
0,1728__3046,1,1728,hoffmann la roche,3046,qiagen gmbh,APA,1,endpoint_only,applicant:hoffmann la roche -> <dummy> -> <dum...,6.0,1.527645,1.000000,0.042768,0.805560,1.230610,0.052631,0
1,1728__3046,1,1728,hoffmann la roche,3046,qiagen gmbh,API,1,endpoint_only,applicant:qiagen gmbh -> <dummy> -> <dummy> ->...,6.0,1.531932,0.066494,1.000000,0.793272,1.215239,0.080806,0
2,1728__3046,1,1728,hoffmann la roche,3046,qiagen gmbh,APC,1,endpoint_only,applicant:qiagen gmbh -> <dummy> -> <dummy> ->...,6.0,1.363664,-0.127960,1.000000,0.722932,0.985837,0.000000,0
3,1728__3046,1,1728,hoffmann la roche,3046,qiagen gmbh,API,2,endpoint_only,applicant:hoffmann la roche -> <dummy> -> <dum...,6.0,1.428405,1.000000,0.066494,0.660267,0.943130,0.062713,0
4,1728__3046,1,1728,hoffmann la roche,3046,qiagen gmbh,APA,2,endpoint_only,applicant:qiagen gmbh -> <dummy> -> <dummy> ->...,6.0,1.388213,0.042768,1.000000,0.626425,0.869611,0.037192,0
5,1728__3046,1,1728,hoffmann la roche,3046,qiagen gmbh,APC,2,endpoint_only,applicant:hoffmann la roche -> <dummy> -> <dum...,6.0,1.262322,1.000000,-0.127960,0.592733,0.748220,0.000000,0


In [39]:
walk_df

,pair_id,rank,source_id,source_name,target_id,target_name,metapath,walk_rank,walk_class,decoded_walk_window,frequency,weighted_influence,support_u,support_v,support_pair,walk_score,balanced_walk_score
0,1728__3046,1,1728,hoffmann la roche,3046,qiagen gmbh,API,1,endpoint_only,applicant:qiagen gmbh -> <dummy> -> <dummy> ->...,6.0,1.531932,0.066494,1.000000,0.793272,1.215239,0.080806
1,1728__3046,1,1728,hoffmann la roche,3046,qiagen gmbh,API,2,endpoint_only,applicant:hoffmann la roche -> <dummy> -> <dum...,6.0,1.428405,1.000000,0.066494,0.660267,0.943130,0.062713
2,1728__3046,1,1728,hoffmann la roche,3046,qiagen gmbh,APA,1,partially_interpretable,applicant:hoffmann la roche -> <dummy> -> <dum...,6.0,1.527645,1.000000,0.042768,0.805560,1.230610,0.052631
3,1728__3046,1,1728,hoffmann la roche,3046,qiagen gmbh,APA,2,partially_interpretable,applicant:qiagen gmbh -> <dummy> -> <dummy> ->...,6.0,1.388213,0.042768,1.000000,0.626425,0.869611,0.037192
4,1728__3046,1,1728,hoffmann la roche,3046,qiagen gmbh,APC,1,endpoint_only,applicant:qiagen gmbh -> <dummy> -> <dummy> ->...,6.0,1.363664,-0.127960,1.000000,0.722932,0.985837,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
100,293__3282,10,293,bayer ag,3282,santaris pharma as,APA,1,partially_interpretable,applicant:santaris pharma as -> <dummy> -> <du...,6.0,1.311628,-0.063842,1.000000,0.607991,0.797458,0.000000
101,293__3282,10,293,bayer ag,3282,santaris pharma as,APA,2,partially_interpretable,applicant:bayer ag -> patent:US 2021/0054979 W...,3.0,0.909919,0.654202,0.180870,0.639616,0.581998,0.306520
102,293__3282,10,293,bayer ag,3282,santaris pharma as,APA,3,partially_interpretable,applicant:bayer ag -> patent:US 202063092973 P...,2.0,0.800017,0.840606,0.103852,0.737174,0.589752,0.155250
103,293__3282,10,293,bayer ag,3282,santaris pharma as,APA,4,partially_interpretable,applicant:bayer ag -> patent:US 202118032096 A...,3.0,0.764192,0.671503,-0.024654,0.517984,0.395839,0.120273
